# 01 — RibFrac Patient-Level Data Loading

## Purpose
<p>This notebook establishes the patient-level data loading pipeline for the
RibFrac chest CT dataset used in the development of the proposed
Enhanced 3D ResUNet-based framework for automatic detection,
segmentation, and classification of rib fractures.</p>

<h4>The notebook performs:</h4>
<ol>
<li> Patient-level dataset management</li>
<li> Automatic retrieval of CT images </li>
<li> Automatic retrieval of corresponding annotations </li>
<li> Association of annotation labels with fracture metadata</li>
<li> CT–annotation spatial validation</li>
<li> Memory-efficient remote access to the RibFrac CT archive </li>
<li> Preparation of validated patient data for downstream preprocessing</li>
<li> Patient-level dataset partitioning to prevent data leakage</li>
</ol>

<p>The CT archive is accessed through HTTP range requests. Individual
patients are retrieved without downloading the complete RibFrac CT archive.</p>

## 1. Import Libraries

In [1]:
#1. Import libraries
import os
import io
import struct
import zipfile
import zlib
import requests
import re
import time

import numpy as np
import pandas as pd
import nibabel as nib
import gc
import traceback
import hashlib
from pathlib import Path
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split

## 2. Define Project Directories

In [2]:
#2. Define project directories
BASE_DIR = Path("/kaggle/working")

CT_DIR = BASE_DIR / "ribfrac_images"
LABEL_DIR = BASE_DIR / "ribfrac_labels"
METADATA_DIR = BASE_DIR / "ribfrac_metadata"

for directory in [CT_DIR, LABEL_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directories:")
print("CT:", CT_DIR)
print("Labels:", LABEL_DIR)
print("Metadata:", METADATA_DIR)

Project directories:
CT: /kaggle/working/ribfrac_images
Labels: /kaggle/working/ribfrac_labels
Metadata: /kaggle/working/ribfrac_metadata


## 3. Define RibFrac Source Information

In [3]:
#3. Define RibFrac source information


# ============================================================
# CELL #3 — DEFINE RIBFRAC SOURCE INFORMATION
# ============================================================

RIBFRAC_SOURCES = {
    "train_part1": {
        "record_id": "3893508",
        "expected_patients": 300,
        "images": "ribfrac-train-images-1.zip",
        "labels": "ribfrac-train-labels-1.zip",
        "info": "ribfrac-train-info-1.csv"
    },

    "train_part2": {
        "record_id": "3893498",
        "expected_patients": 120,
        "images": "ribfrac-train-images-2.zip",
        "labels": "ribfrac-train-labels-2.zip",
        "info": "ribfrac-train-info-2.csv"
    },

    "validation": {
        "record_id": "3893496",
        "expected_patients": 80,
        "images": "ribfrac-val-images.zip",
        "labels": "ribfrac-val-labels.zip",
        "info": "ribfrac-val-info.csv"
    }
}

# Generate source URLs
for name, source in RIBFRAC_SOURCES.items():

    base_url = (
        f"https://zenodo.org/records/"
        f"{source['record_id']}/files/"
    )

    source["ct_url"] = (
        base_url + source["images"] + "?download=1"
    )

    source["label_url"] = (
        base_url + source["labels"] + "?download=1"
    )

    source["info_url"] = (
        base_url + source["info"] + "?download=1"
    )

# Keep original Part 1 aliases for existing functions
ZENODO_RECORD_PART1 = (
    "https://zenodo.org/records/3893508"
)

CT_ZIP_URL = RIBFRAC_SOURCES["train_part1"]["ct_url"]
LABEL_ZIP_URL = RIBFRAC_SOURCES["train_part1"]["label_url"]
INFO_URL = RIBFRAC_SOURCES["train_part1"]["info_url"]

total = sum(
    source["expected_patients"]
    for source in RIBFRAC_SOURCES.values()
)

print("RibFrac sources configured successfully.")

for name, source in RIBFRAC_SOURCES.items():
    print(
        name,
        ":", source["expected_patients"], "patients"
    )

print("Total expected labelled patients:", total)

assert total == 500


RibFrac sources configured successfully.
train_part1 : 300 patients
train_part2 : 120 patients
validation : 80 patients
Total expected labelled patients: 500


## 4. Download metadata

In [4]:
#4. Download metadata

# ============================================================
# CELL #4 — RIBFRAC METADATA CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# 1. Metadata directory
# ------------------------------------------------------------

BASE_DIR = Path("/kaggle/working")

METADATA_DIR = BASE_DIR / "ribfrac_metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Configure metadata for all RibFrac sources
#    Uses RIBFRAC_SOURCES defined in Cell #3
# ------------------------------------------------------------

METADATA_SOURCES = {}

for source_name, source_info in RIBFRAC_SOURCES.items():

    csv_filename = source_info["info"]

    METADATA_SOURCES[source_name] = {
        "filename": csv_filename,
        "path": METADATA_DIR / csv_filename,
        "url": source_info["info_url"],
        "expected_patients": source_info["expected_patients"]
    }

# ------------------------------------------------------------
# 3. Metadata paths for all three source partitions
# ------------------------------------------------------------

INFO_PATHS = {
    name: config["path"]
    for name, config in METADATA_SOURCES.items()
}

INFO_URLS = {
    name: config["url"]
    for name, config in METADATA_SOURCES.items()
}

# Preserve original Part 1 aliases for older notebook cells
INFO_PATH = INFO_PATHS["train_part1"]
INFO_URL = INFO_URLS["train_part1"]

# ------------------------------------------------------------
# 4. Define combined metadata output
# ------------------------------------------------------------

COMBINED_INFO_PATH = (
    METADATA_DIR / "ribfrac-info-500-combined.csv"
)

# ------------------------------------------------------------
# 5. Verify metadata configuration
# ------------------------------------------------------------

EXPECTED_PATIENTS = sum(
    config["expected_patients"]
    for config in METADATA_SOURCES.values()
)

assert len(METADATA_SOURCES) == 3
assert EXPECTED_PATIENTS == 500

print("RibFrac metadata configuration completed.")

for name, config in METADATA_SOURCES.items():

    print(f"\nSource: {name}")
    print(f"Expected patients: {config['expected_patients']}")
    print(f"Filename: {config['filename']}")
    print(f"Local path: {config['path']}")
    print(f"URL: {config['url']}")

print("\nTotal expected patients:", EXPECTED_PATIENTS)
print("Combined metadata path:", COMBINED_INFO_PATH)

print("\nCELL #4 CONFIGURATION: PASS")


RibFrac metadata configuration completed.

Source: train_part1
Expected patients: 300
Filename: ribfrac-train-info-1.csv
Local path: /kaggle/working/ribfrac_metadata/ribfrac-train-info-1.csv
URL: https://zenodo.org/records/3893508/files/ribfrac-train-info-1.csv?download=1

Source: train_part2
Expected patients: 120
Filename: ribfrac-train-info-2.csv
Local path: /kaggle/working/ribfrac_metadata/ribfrac-train-info-2.csv
URL: https://zenodo.org/records/3893498/files/ribfrac-train-info-2.csv?download=1

Source: validation
Expected patients: 80
Filename: ribfrac-val-info.csv
Local path: /kaggle/working/ribfrac_metadata/ribfrac-val-info.csv
URL: https://zenodo.org/records/3893496/files/ribfrac-val-info.csv?download=1

Total expected patients: 500
Combined metadata path: /kaggle/working/ribfrac_metadata/ribfrac-info-500-combined.csv

CELL #4 CONFIGURATION: PASS


## 5. Load Metadata

In [5]:

# ============================================================
# CELL #5 — LOAD AND COMBINE RIBFRAC METADATA
# ============================================================

metadata_frames = []

REQUIRED_COLUMNS = [
    "public_id",
    "label_id",
    "label_code"
]

# ------------------------------------------------------------
# 1. Download and load metadata from all three sources
# ------------------------------------------------------------

for source_name, config in METADATA_SOURCES.items():

    csv_path = config["path"]
    csv_url = config["url"]

    print(f"\nProcessing source: {source_name}")

    # Download only if the file does not already exist
    if not csv_path.exists():

        print("Downloading metadata...")

        response = requests.get(
            csv_url,
            timeout=120
        )

        response.raise_for_status()

        csv_path.write_bytes(response.content)

        print("Download completed.")

    else:
        print("Existing metadata file found.")

    # Load CSV
    df = pd.read_csv(csv_path)

    # Validate required columns
    missing = set(REQUIRED_COLUMNS) - set(df.columns)

    if missing:
        raise ValueError(
            f"{source_name}: Missing columns {missing}"
        )

    # Keep required metadata columns
    df = df[REQUIRED_COLUMNS].copy()

    df["public_id"] = df["public_id"].astype(str)

    df["label_id"] = pd.to_numeric(
        df["label_id"],
        errors="raise"
    ).astype(int)

    df["label_code"] = pd.to_numeric(
        df["label_code"],
        errors="raise"
    ).astype(int)

    # Preserve original dataset partition
    df["source_partition"] = source_name

    # Verify number of unique patients
    patients = df["public_id"].nunique()

    expected = config["expected_patients"]

    print("Metadata records:", len(df))
    print("Unique patients:", patients)

    assert patients == expected, (
        f"{source_name}: Expected {expected} patients, "
        f"found {patients}"
    )

    metadata_frames.append(df)

# ------------------------------------------------------------
# 2. Combine metadata
# ------------------------------------------------------------

info_df = pd.concat(
    metadata_frames,
    ignore_index=True
)

# ------------------------------------------------------------
# 3. Validate combined dataset
# ------------------------------------------------------------

total_patients = info_df["public_id"].nunique()

assert total_patients == 500, (
    f"Expected 500 unique patients, "
    f"found {total_patients}"
)

# No patient should appear in multiple source partitions
patient_sources = (
    info_df.groupby("public_id")["source_partition"]
    .nunique()
)

assert patient_sources.max() == 1, (
    "A patient appears in multiple source partitions."
)

# No duplicate fracture-instance metadata keys
assert not info_df.duplicated(
    subset=["public_id", "label_id"]
).any(), "Duplicate patient/annotation identifiers found."

# ------------------------------------------------------------
# 4. Save combined metadata
# ------------------------------------------------------------

info_df.to_csv(
    COMBINED_INFO_PATH,
    index=False
)

# ------------------------------------------------------------
# 5. Display verification results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("RIBFRAC METADATA LOADED SUCCESSFULLY")
print("=" * 55)

print("Combined metadata shape:", info_df.shape)

print("Total unique patients:", total_patients)

print("\nPatients by source:")

print(
    info_df.groupby("source_partition")["public_id"]
    .nunique()
)

print("\nMetadata columns:")
print(info_df.columns.tolist())

print("\nFracture label code distribution:")
print(info_df["label_code"].value_counts().sort_index())

print("\nFirst 10 metadata records:")
display(info_df.head(10))

print("\nSaved combined metadata:")
print(COMBINED_INFO_PATH)

print("\nCELL #5 VALIDATION: PASS")



Processing source: train_part1
Download completed.
Metadata records: 3152
Unique patients: 300

Processing source: train_part2
Download completed.
Metadata records: 1255
Unique patients: 120

Processing source: validation
Download completed.
Metadata records: 515
Unique patients: 80

RIBFRAC METADATA LOADED SUCCESSFULLY
Combined metadata shape: (4922, 4)
Total unique patients: 500

Patients by source:
source_partition
train_part1    300
train_part2    120
validation      80
Name: public_id, dtype: int64

Metadata columns:
['public_id', 'label_id', 'label_code', 'source_partition']

Fracture label code distribution:
label_code
-1    2575
 0     500
 1     687
 2     630
 3     321
 4     209
Name: count, dtype: int64

First 10 metadata records:


,public_id,label_id,label_code,source_partition
0,RibFrac1,0,0,train_part1
1,RibFrac1,1,2,train_part1
2,RibFrac1,2,2,train_part1
3,RibFrac2,0,0,train_part1
4,RibFrac2,1,-1,train_part1
5,RibFrac2,2,-1,train_part1
6,RibFrac2,3,-1,train_part1
7,RibFrac2,4,2,train_part1
8,RibFrac2,5,2,train_part1
9,RibFrac2,6,2,train_part1



Saved combined metadata:
/kaggle/working/ribfrac_metadata/ribfrac-info-500-combined.csv

CELL #5 VALIDATION: PASS


## 6. Examine patients

In [6]:
#6. Examine patients

# ============================================================
# CELL #6 — EXAMINE ALL RIBFRAC PATIENTS
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Verify combined metadata
# ------------------------------------------------------------

required_columns = [
    "public_id",
    "label_id",
    "label_code",
    "source_partition"
]

missing_columns = [
    col for col in required_columns
    if col not in info_df.columns
]

assert not missing_columns, (
    f"Missing metadata columns: {missing_columns}"
)

print("Combined metadata shape:", info_df.shape)

# ------------------------------------------------------------
# 2. Extract unique patient identifiers
# ------------------------------------------------------------

patient_ids = sorted(
    info_df["public_id"].dropna().unique().tolist()
)

total_patients = len(patient_ids)

print("\nTotal unique patients:", total_patients)

print("\nFirst 20 patients:")
print(patient_ids[:20])

print("\nLast 20 patients:")
print(patient_ids[-20:])

# ------------------------------------------------------------
# 3. Create patient-level source registry
# ------------------------------------------------------------

patient_registry = (
    info_df[["public_id", "source_partition"]]
    .drop_duplicates()
    .sort_values("public_id")
    .reset_index(drop=True)
)

# Verify each patient belongs to exactly one source
assert not patient_registry["public_id"].duplicated().any(), (
    "Duplicate patient found across source partitions."
)

assert patient_registry["source_partition"].notna().all()

# ------------------------------------------------------------
# 4. Count patients by source
# ------------------------------------------------------------

source_counts = (
    patient_registry["source_partition"]
    .value_counts()
)

expected_counts = {
    "train_part1": 300,
    "train_part2": 120,
    "validation": 80
}

print("\nPatients by source:")

for source, expected in expected_counts.items():

    actual = int(source_counts.get(source, 0))

    print(
        f"{source}: {actual} "
        f"(expected {expected})"
    )

    assert actual == expected, (
        f"{source}: Expected {expected}, "
        f"found {actual}"
    )

assert set(source_counts.index) == set(expected_counts)

# ------------------------------------------------------------
# 5. Verify complete patient population
# ------------------------------------------------------------

assert total_patients == 500, (
    f"Expected 500 patients, found {total_patients}"
)

assert len(patient_registry) == 500

# ------------------------------------------------------------
# 6. Check annotation counts
# ------------------------------------------------------------

annotations_per_patient = (
    info_df.groupby("public_id")
    .size()
)

print("\nAnnotation records per patient:")

print(annotations_per_patient.describe())

print("\nFracture subtype code distribution:")

print(
    info_df["label_code"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 7. Save patient-source registry
# ------------------------------------------------------------

PATIENT_REGISTRY_PATH = (
    METADATA_DIR / "patient_source_registry_500.csv"
)

patient_registry.to_csv(
    PATIENT_REGISTRY_PATH,
    index=False
)

# Create patient-to-source lookup for later cells
patient_source_map = (
    patient_registry
    .set_index("public_id")["source_partition"]
    .to_dict()
)

# ------------------------------------------------------------
# 8. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("RIBFRAC PATIENT EXAMINATION COMPLETED")
print("=" * 55)

print("Total unique patients:", total_patients)
print("Total metadata records:", len(info_df))
print("Patient registry saved:", PATIENT_REGISTRY_PATH)

print("\nCELL #6 VALIDATION: PASS")


Combined metadata shape: (4922, 4)

Total unique patients: 500

First 20 patients:
['RibFrac1', 'RibFrac10', 'RibFrac100', 'RibFrac101', 'RibFrac102', 'RibFrac103', 'RibFrac104', 'RibFrac105', 'RibFrac106', 'RibFrac107', 'RibFrac108', 'RibFrac109', 'RibFrac11', 'RibFrac110', 'RibFrac111', 'RibFrac112', 'RibFrac113', 'RibFrac114', 'RibFrac115', 'RibFrac116']

Last 20 patients:
['RibFrac81', 'RibFrac82', 'RibFrac83', 'RibFrac84', 'RibFrac85', 'RibFrac86', 'RibFrac87', 'RibFrac88', 'RibFrac89', 'RibFrac9', 'RibFrac90', 'RibFrac91', 'RibFrac92', 'RibFrac93', 'RibFrac94', 'RibFrac95', 'RibFrac96', 'RibFrac97', 'RibFrac98', 'RibFrac99']

Patients by source:
train_part1: 300 (expected 300)
train_part2: 120 (expected 120)
validation: 80 (expected 80)

Annotation records per patient:
count    500.000000
mean       9.844000
std        5.700584
min        1.000000
25%        6.000000
50%        9.000000
75%       13.000000
max       37.000000
dtype: float64

Fracture subtype code distribution:
la

## 7. Build patient-level annotation table

In [7]:
#7. Build patient-level annotation table
patient_annotations = (
    info_df
    .groupby("public_id")
    .agg(
        number_of_annotations=("label_id", "count"),
        label_codes=("label_code", lambda x: sorted(x.dropna().unique().tolist()))
    )
    .reset_index()
)

print("Patient-level annotation table:")
display(patient_annotations.head(10))

Patient-level annotation table:


,public_id,number_of_annotations,label_codes
0,RibFrac1,3,"[0, 2]"
1,RibFrac10,9,"[-1, 0, 1, 2, 3]"
2,RibFrac100,13,"[-1, 0, 1, 3]"
3,RibFrac101,5,"[-1, 0, 1, 2]"
4,RibFrac102,18,"[-1, 0, 2, 3, 4]"
5,RibFrac103,12,"[-1, 0, 2]"
6,RibFrac104,14,"[-1, 0]"
7,RibFrac105,11,"[-1, 0, 1, 2]"
8,RibFrac106,13,"[-1, 0, 1]"
9,RibFrac107,7,"[-1, 0, 1]"


## 8. Count fracture codes

In [8]:

# ============================================================
# CELL #7 — BUILD PATIENT-LEVEL ANNOTATION TABLE
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Verify input metadata
# ------------------------------------------------------------

required_columns = [
    "public_id",
    "label_id",
    "label_code",
    "source_partition"
]

assert all(
    col in info_df.columns
    for col in required_columns
), "Required metadata columns are missing."

assert info_df["public_id"].nunique() == 500

# ------------------------------------------------------------
# 2. Prepare annotation information
# ------------------------------------------------------------

annotation_df = info_df.copy()

# label_id = 0 represents background
# Positive label IDs identify fracture instances
annotation_df["is_fracture"] = (
    annotation_df["label_id"] > 0
)

# Unknown fracture subtype remains a fracture
annotation_df["is_unknown_subtype"] = (
    (annotation_df["label_id"] > 0) &
    (annotation_df["label_code"] == -1)
)

# Known fracture subtype classes: 1, 2, 3, 4
annotation_df["is_classified_fracture"] = (
    (annotation_df["label_id"] > 0) &
    (annotation_df["label_code"].isin([1, 2, 3, 4]))
)

# ------------------------------------------------------------
# 3. Build patient-level annotation table
# ------------------------------------------------------------

patient_annotations = (
    annotation_df
    .groupby(
        ["public_id", "source_partition"],
        as_index=False
    )
    .agg(
        number_of_annotations=(
            "label_id", "count"
        ),
        number_of_fractures=(
            "is_fracture", "sum"
        ),
        number_of_unknown_subtypes=(
            "is_unknown_subtype", "sum"
        ),
        number_of_classified_fractures=(
            "is_classified_fracture", "sum"
        ),
        label_codes=(
            "label_code",
            lambda x: sorted(x.dropna().unique().tolist())
        )
    )
)

# ------------------------------------------------------------
# 4. Validate patient-level table
# ------------------------------------------------------------

assert len(patient_annotations) == 500, (
    f"Expected 500 patients, "
    f"found {len(patient_annotations)}"
)

assert (
    patient_annotations["public_id"].nunique() == 500
), "Duplicate patient IDs detected."

assert (
    patient_annotations["number_of_fractures"] ==
    patient_annotations["number_of_unknown_subtypes"] +
    patient_annotations["number_of_classified_fractures"]
).all(), (
    "Some fracture instances have unsupported subtype codes."
)

# ------------------------------------------------------------
# 5. Display patient-level information
# ------------------------------------------------------------

print("=" * 60)
print("RIBFRAC PATIENT-LEVEL ANNOTATION TABLE")
print("=" * 60)

print("\nTotal patients:", len(patient_annotations))

print("\nPatients by source:")
print(
    patient_annotations["source_partition"]
    .value_counts()
)

print(
    "\nTotal metadata records:",
    patient_annotations["number_of_annotations"].sum()
)

print(
    "Total fracture instances:",
    patient_annotations["number_of_fractures"].sum()
)

print(
    "Classified fractures:",
    patient_annotations[
        "number_of_classified_fractures"
    ].sum()
)

print(
    "Fractures with unknown subtype:",
    patient_annotations[
        "number_of_unknown_subtypes"
    ].sum()
)

print("\nFirst 10 patient records:")
display(patient_annotations.head(10))

# ------------------------------------------------------------
# 6. Save patient-level annotation table
# ------------------------------------------------------------

PATIENT_ANNOTATION_PATH = (
    METADATA_DIR / "patient_annotation_table_500.csv"
)

patient_annotations.to_csv(
    PATIENT_ANNOTATION_PATH,
    index=False
)

print("\nPatient annotation table saved:")
print(PATIENT_ANNOTATION_PATH)

print("\nCELL #7 VALIDATION: PASS")


RIBFRAC PATIENT-LEVEL ANNOTATION TABLE

Total patients: 500

Patients by source:
source_partition
train_part1    300
train_part2    120
validation      80
Name: count, dtype: int64

Total metadata records: 4922
Total fracture instances: 4422
Classified fractures: 1847
Fractures with unknown subtype: 2575

First 10 patient records:


,public_id,source_partition,number_of_annotations,number_of_fractures,number_of_unknown_subtypes,number_of_classified_fractures,label_codes
0,RibFrac1,train_part1,3,2,0,2,"[0, 2]"
1,RibFrac10,train_part1,9,8,3,5,"[-1, 0, 1, 2, 3]"
2,RibFrac100,train_part1,13,12,9,3,"[-1, 0, 1, 3]"
3,RibFrac101,train_part1,5,4,1,3,"[-1, 0, 1, 2]"
4,RibFrac102,train_part1,18,17,9,8,"[-1, 0, 2, 3, 4]"
5,RibFrac103,train_part1,12,11,9,2,"[-1, 0, 2]"
6,RibFrac104,train_part1,14,13,13,0,"[-1, 0]"
7,RibFrac105,train_part1,11,10,5,5,"[-1, 0, 1, 2]"
8,RibFrac106,train_part1,13,12,11,1,"[-1, 0, 1]"
9,RibFrac107,train_part1,7,6,5,1,"[-1, 0, 1]"



Patient annotation table saved:
/kaggle/working/ribfrac_metadata/patient_annotation_table_500.csv

CELL #7 VALIDATION: PASS


## 8 — COUNT RIBFRAC FRACTURE CODES

In [9]:

# ============================================================
# CELL #8 — COUNT RIBFRAC FRACTURE CODES
# ============================================================

# ------------------------------------------------------------
# 1. Verify metadata
# ------------------------------------------------------------

required_columns = [
    "public_id",
    "label_id",
    "label_code",
    "source_partition"
]

assert all(
    col in info_df.columns
    for col in required_columns
), "Required metadata columns are missing."

assert info_df["public_id"].nunique() == 500

# ------------------------------------------------------------
# 2. Define RibFrac fracture categories
# ------------------------------------------------------------

FRACTURE_CLASSES = {
    -1: "Unknown subtype",
     0: "Background",
     1: "Displaced fracture",
     2: "Nondisplaced fracture",
     3: "Buckle fracture",
     4: "Segmental fracture"
}

# ------------------------------------------------------------
# 3. Count annotation label codes
# ------------------------------------------------------------

fracture_counts = (
    info_df["label_code"]
    .value_counts(dropna=False)
    .sort_index()
)

print("=" * 60)
print("RIBFRAC FRACTURE LABEL DISTRIBUTION")
print("=" * 60)

print("\nRaw label-code counts:")
print(fracture_counts)

# ------------------------------------------------------------
# 4. Build labelled distribution table
# ------------------------------------------------------------

code_distribution = (
    info_df.groupby("label_code", dropna=False)
    .size()
    .reset_index(name="count")
)

code_distribution["fracture_type"] = (
    code_distribution["label_code"]
    .map(FRACTURE_CLASSES)
    .fillna("Unrecognized code")
)

total_records = len(info_df)

code_distribution["percentage"] = (
    code_distribution["count"]
    / total_records * 100
).round(2)

code_distribution = code_distribution[
    [
        "label_code",
        "fracture_type",
        "count",
        "percentage"
    ]
].sort_values("label_code")

print("\nAnnotation label distribution:")
display(code_distribution)

# ------------------------------------------------------------
# 5. Count actual fracture-instance records
# ------------------------------------------------------------

# Fracture instances are identified by positive label IDs.
# A subtype code of -1 does not remove a fracture.

fracture_df = info_df[
    info_df["label_id"] > 0
].copy()

total_fractures = len(fracture_df)

known_fractures = (
    fracture_df["label_code"]
    .isin([1, 2, 3, 4])
    .sum()
)

unknown_fractures = (
    fracture_df["label_code"] == -1
).sum()

print("\nFracture-instance summary:")

print("Total patients:", info_df["public_id"].nunique())
print("Total metadata records:", total_records)
print("Total fracture instances:", total_fractures)
print("Known subtype fractures:", known_fractures)
print("Unknown subtype fractures:", unknown_fractures)

# ------------------------------------------------------------
# 6. Fracture-code distribution by dataset source
# ------------------------------------------------------------

source_distribution = pd.crosstab(
    info_df["source_partition"],
    info_df["label_code"]
)

print("\nFracture codes by dataset source:")
display(source_distribution)

# ------------------------------------------------------------
# 7. Patients with each fracture subtype
# ------------------------------------------------------------

patients_by_type = (
    fracture_df
    .groupby("label_code")["public_id"]
    .nunique()
    .reset_index(name="number_of_patients")
)

patients_by_type["fracture_type"] = (
    patients_by_type["label_code"]
    .map(FRACTURE_CLASSES)
)

print("\nPatients with each fracture subtype:")
display(patients_by_type)

# ------------------------------------------------------------
# 8. Check annotation-code consistency
# ------------------------------------------------------------

valid_codes = set(FRACTURE_CLASSES.keys())

observed_codes = set(
    info_df["label_code"].dropna().unique()
)

unexpected_codes = observed_codes - valid_codes

# A positive label ID should not have background code 0.
inconsistent_records = info_df[
    (info_df["label_id"] > 0) &
    (info_df["label_code"] == 0)
]

print("\nUnexpected codes:", unexpected_codes)
print(
    "Positive label IDs with background subtype:",
    len(inconsistent_records)
)

assert info_df["label_code"].notna().all(), (
    "Missing fracture subtype codes detected."
)

assert not unexpected_codes, (
    f"Unexpected fracture codes: {unexpected_codes}"
)

assert inconsistent_records.empty, (
    "Some positive fracture labels have background code 0."
)

assert (
    total_fractures ==
    known_fractures + unknown_fractures
), "Fracture subtype count mismatch."

# ------------------------------------------------------------
# 9. Save distribution tables
# ------------------------------------------------------------

CODE_DISTRIBUTION_PATH = (
    METADATA_DIR / "ribfrac_fracture_code_distribution.csv"
)

SOURCE_DISTRIBUTION_PATH = (
    METADATA_DIR / "ribfrac_fracture_codes_by_source.csv"
)

code_distribution.to_csv(
    CODE_DISTRIBUTION_PATH,
    index=False
)

source_distribution.to_csv(
    SOURCE_DISTRIBUTION_PATH
)

# ------------------------------------------------------------
# 10. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RIBFRAC FRACTURE CODE ANALYSIS COMPLETED")
print("=" * 60)

print("Total patients:", info_df["public_id"].nunique())
print("Total fracture instances:", total_fractures)
print("Distribution tables saved.")

print("\nCELL #8 VALIDATION: PASS")


RIBFRAC FRACTURE LABEL DISTRIBUTION

Raw label-code counts:
label_code
-1    2575
 0     500
 1     687
 2     630
 3     321
 4     209
Name: count, dtype: int64

Annotation label distribution:


,label_code,fracture_type,count,percentage
0,-1,Unknown subtype,2575,52.32
1,0,Background,500,10.16
2,1,Displaced fracture,687,13.96
3,2,Nondisplaced fracture,630,12.80
4,3,Buckle fracture,321,6.52
5,4,Segmental fracture,209,4.25



Fracture-instance summary:
Total patients: 500
Total metadata records: 4922
Total fracture instances: 4422
Known subtype fractures: 1847
Unknown subtype fractures: 2575

Fracture codes by dataset source:


label_code,-1,0,1,2,3,4
source_partition,,,,,,
train_part1,1646,300,450,417,197,142
train_part2,686,120,168,150,94,37
validation,243,80,69,63,30,30



Patients with each fracture subtype:


,label_code,number_of_patients,fracture_type
0,-1,431,Unknown subtype
1,1,270,Displaced fracture
2,2,246,Nondisplaced fracture
3,3,152,Buckle fracture
4,4,76,Segmental fracture



Unexpected codes: set()
Positive label IDs with background subtype: 0

RIBFRAC FRACTURE CODE ANALYSIS COMPLETED
Total patients: 500
Total fracture instances: 4422
Distribution tables saved.

CELL #8 VALIDATION: PASS


## 9. Download the label archive if necessary

In [10]:

# ============================================================
# CELL #9 — DOWNLOAD AND VERIFY RIBFRAC LABEL ARCHIVES
# ============================================================
# ------------------------------------------------------------
# 1. Verify configuration
# ------------------------------------------------------------

assert len(RIBFRAC_SOURCES) == 3
assert info_df["public_id"].nunique() == 500

LABEL_ARCHIVE_DIR = BASE_DIR / "ribfrac_label_archives"
LABEL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ZIP_PATHS = {}
label_archive_members = {}

# ------------------------------------------------------------
# 2. Download each annotation archive if necessary
# ------------------------------------------------------------

for source_name, source in RIBFRAC_SOURCES.items():

    filename = source["labels"]
    download_url = source["label_url"]

    zip_path = LABEL_ARCHIVE_DIR / filename

    print("\n" + "=" * 60)
    print(f"Processing: {source_name}")
    print("=" * 60)

    if not zip_path.exists() or zip_path.stat().st_size == 0:

        print("Downloading:", filename)

        temporary_path = zip_path.with_suffix(".zip.part")

        try:
            with requests.get(
                download_url,
                stream=True,
                timeout=(30, 180)
            ) as response:

                response.raise_for_status()

                with open(temporary_path, "wb") as file:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            file.write(chunk)

            temporary_path.replace(zip_path)

            print("Download completed.")

        except Exception:
            temporary_path.unlink(missing_ok=True)
            raise

    else:
        print("Existing archive found:", filename)

    # --------------------------------------------------------
    # 3. Validate ZIP archive integrity
    # --------------------------------------------------------

    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError(
            f"Invalid ZIP archive: {zip_path}"
        )

    with zipfile.ZipFile(zip_path, "r") as archive:

        bad_file = archive.testzip()

        if bad_file is not None:
            raise RuntimeError(
                f"Corrupted ZIP member: {bad_file}"
            )

        members = [
            name
            for name in archive.namelist()
            if name.endswith("-label.nii.gz")
        ]

    if len(members) != len(set(members)):
        raise RuntimeError(
            f"Duplicate ZIP members in {source_name}"
        )

    # --------------------------------------------------------
    # 4. Verify patients against source metadata
    # --------------------------------------------------------

    archive_patients = {
        Path(member).name.replace("-label.nii.gz", "")
        for member in members
    }

    metadata_patients = set(
        info_df.loc[
            info_df["source_partition"] == source_name,
            "public_id"
        ]
    )

    expected = source["expected_patients"]

    print("Expected patients:", expected)
    print("Label files:", len(members))
    print("Unique label patients:", len(archive_patients))

    if len(members) != expected:
        raise RuntimeError(
            f"{source_name}: Expected {expected} label files, "
            f"found {len(members)}"
        )

    if archive_patients != metadata_patients:
        raise RuntimeError(
            f"{source_name}: Label/metadata mismatch.\n"
            f"Missing labels: "
            f"{sorted(metadata_patients - archive_patients)}\n"
            f"Unexpected labels: "
            f"{sorted(archive_patients - metadata_patients)}"
        )

    LABEL_ZIP_PATHS[source_name] = zip_path

    label_archive_members[source_name] = members

    print("ZIP integrity: PASS")
    print("Metadata matching: PASS")

# ------------------------------------------------------------
# 5. Verify all three archives together
# ------------------------------------------------------------

all_label_patients = []

for source_name, members in label_archive_members.items():

    for member in members:

        patient_id = (
            Path(member).name.replace("-label.nii.gz", "")
        )

        all_label_patients.append(patient_id)

assert len(all_label_patients) == 500

assert len(set(all_label_patients)) == 500, (
    "Duplicate patient IDs across label archives."
)

assert set(all_label_patients) == set(patient_ids)

# Legacy alias for Part 1.
# Later cells must use LABEL_ZIP_PATHS for all 500 patients.
LABEL_ZIP_PATH = LABEL_ZIP_PATHS["train_part1"]

# ------------------------------------------------------------
# 6. Final report
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RIBFRAC LABEL ARCHIVE VERIFICATION COMPLETED")
print("=" * 60)

for source_name, zip_path in LABEL_ZIP_PATHS.items():

    print(
        f"{source_name}: "
        f"{len(label_archive_members[source_name])} labels"
    )

    print("Archive:", zip_path)

print("\nTotal unique patients:", len(set(all_label_patients)))

print("\nCELL #9 VALIDATION: PASS")



Processing: train_part1
Downloading: ribfrac-train-labels-1.zip
Download completed.
Expected patients: 300
Label files: 300
Unique label patients: 300
ZIP integrity: PASS
Metadata matching: PASS

Processing: train_part2
Downloading: ribfrac-train-labels-2.zip
Download completed.
Expected patients: 120
Label files: 120
Unique label patients: 120
ZIP integrity: PASS
Metadata matching: PASS

Processing: validation
Downloading: ribfrac-val-labels.zip
Download completed.
Expected patients: 80
Label files: 80
Unique label patients: 80
ZIP integrity: PASS
Metadata matching: PASS

RIBFRAC LABEL ARCHIVE VERIFICATION COMPLETED
train_part1: 300 labels
Archive: /kaggle/working/ribfrac_label_archives/ribfrac-train-labels-1.zip
train_part2: 120 labels
Archive: /kaggle/working/ribfrac_label_archives/ribfrac-train-labels-2.zip
validation: 80 labels
Archive: /kaggle/working/ribfrac_label_archives/ribfrac-val-labels.zip

Total unique patients: 500

CELL #9 VALIDATION: PASS


## 10. Build a label-file index

In [11]:

# ============================================================
# CELL #10 — BUILD RIBFRAC LABEL-FILE INDEX
# ============================================================

import pandas as pd
import zipfile
from pathlib import Path

# ------------------------------------------------------------
# 1. Verify outputs from Cell #9
# ------------------------------------------------------------

assert len(LABEL_ZIP_PATHS) == 3, (
    "Expected three label archives."
)

assert len(label_archive_members) == 3, (
    "Missing label archive member lists."
)

assert len(patient_source_map) == 500, (
    "Patient source registry is incomplete."
)

# ------------------------------------------------------------
# 2. Initialize label indexing structures
# ------------------------------------------------------------

label_members_by_source = {}
label_records = []

# ------------------------------------------------------------
# 3. Index annotation files in each archive
# ------------------------------------------------------------

for source_name, zip_path in LABEL_ZIP_PATHS.items():

    print("\n" + "=" * 60)
    print("Indexing source:", source_name)
    print("=" * 60)

    with zipfile.ZipFile(zip_path, "r") as archive:

        members = sorted([
            member
            for member in archive.namelist()
            if member.endswith("-label.nii.gz")
            and not member.startswith("__MACOSX/")
        ])

    label_members_by_source[source_name] = members

    for member in members:

        filename = Path(member).name

        patient_id = filename.removesuffix(
            "-label.nii.gz"
        )

        label_records.append({
            "patient_id": patient_id,
            "source_partition": source_name,
            "archive_path": str(zip_path),
            "label_member": member,
            "label_filename": filename
        })

    print("Label files indexed:", len(members))

    print("\nFirst five files:")

    for member in members[:5]:
        print(member)

# ------------------------------------------------------------
# 4. Build consolidated label index table
# ------------------------------------------------------------

label_index_df = pd.DataFrame(label_records)

label_index_df = (
    label_index_df
    .sort_values(["source_partition", "patient_id"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Verify unique patient identifiers
# ------------------------------------------------------------

duplicate_patients = label_index_df[
    label_index_df["patient_id"].duplicated(
        keep=False
    )
]

assert duplicate_patients.empty, (
    "Duplicate patient IDs detected in label archives."
)

assert len(label_index_df) == 500, (
    f"Expected 500 label files, "
    f"found {len(label_index_df)}"
)

assert label_index_df["patient_id"].nunique() == 500

# ------------------------------------------------------------
# 6. Verify consistency with patient metadata
# ------------------------------------------------------------

metadata_patients = set(patient_ids)

indexed_patients = set(
    label_index_df["patient_id"]
)

missing_patients = (
    metadata_patients - indexed_patients
)

unexpected_patients = (
    indexed_patients - metadata_patients
)

print("\nMissing label patients:", len(missing_patients))
print("Unexpected label patients:", len(unexpected_patients))

assert not missing_patients, (
    f"Missing labels: {sorted(missing_patients)}"
)

assert not unexpected_patients, (
    f"Unexpected patients: {sorted(unexpected_patients)}"
)

# Verify source partition for every patient
source_mismatches = label_index_df[
    label_index_df.apply(
        lambda row: (
            patient_source_map.get(row["patient_id"])
            != row["source_partition"]
        ),
        axis=1
    )
]

assert source_mismatches.empty, (
    "Patient source partition mismatch detected."
)

# ------------------------------------------------------------
# 7. Save consolidated index
# ------------------------------------------------------------

LABEL_INDEX_PATH = (
    METADATA_DIR / "ribfrac_label_file_index_500.csv"
)

label_index_df.to_csv(
    LABEL_INDEX_PATH,
    index=False
)

# Preserve a flat file list for compatibility
# Archive information must still be used when extracting.
label_members = label_index_df[
    "label_member"
].tolist()

# ------------------------------------------------------------
# 8. Display final index summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RIBFRAC LABEL-FILE INDEX COMPLETED")
print("=" * 60)

print("\nLabel files by source:")

print(
    label_index_df["source_partition"]
    .value_counts()
    .sort_index()
)

print("\nTotal label files:", len(label_index_df))

print(
    "Unique patients:",
    label_index_df["patient_id"].nunique()
)

print("\nFirst 10 indexed label files:")

display(label_index_df.head(10))

print("\nIndex saved to:")
print(LABEL_INDEX_PATH)

print("\nCELL #10 VALIDATION: PASS")



Indexing source: train_part1
Label files indexed: 300

First five files:
Part1/RibFrac1-label.nii.gz
Part1/RibFrac10-label.nii.gz
Part1/RibFrac100-label.nii.gz
Part1/RibFrac101-label.nii.gz
Part1/RibFrac102-label.nii.gz

Indexing source: train_part2
Label files indexed: 120

First five files:
Part2/RibFrac301-label.nii.gz
Part2/RibFrac302-label.nii.gz
Part2/RibFrac303-label.nii.gz
Part2/RibFrac304-label.nii.gz
Part2/RibFrac305-label.nii.gz

Indexing source: validation
Label files indexed: 80

First five files:
ribfrac-val-labels/RibFrac421-label.nii.gz
ribfrac-val-labels/RibFrac422-label.nii.gz
ribfrac-val-labels/RibFrac423-label.nii.gz
ribfrac-val-labels/RibFrac424-label.nii.gz
ribfrac-val-labels/RibFrac425-label.nii.gz

Missing label patients: 0
Unexpected label patients: 0

RIBFRAC LABEL-FILE INDEX COMPLETED

Label files by source:
source_partition
train_part1    300
train_part2    120
validation      80
Name: count, dtype: int64

Total label files: 500
Unique patients: 500

First 

,patient_id,source_partition,archive_path,label_member,label_filename
0,RibFrac1,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac1-label.nii.gz,RibFrac1-label.nii.gz
1,RibFrac10,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac10-label.nii.gz,RibFrac10-label.nii.gz
2,RibFrac100,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac100-label.nii.gz,RibFrac100-label.nii.gz
3,RibFrac101,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac101-label.nii.gz,RibFrac101-label.nii.gz
4,RibFrac102,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac102-label.nii.gz,RibFrac102-label.nii.gz
5,RibFrac103,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac103-label.nii.gz,RibFrac103-label.nii.gz
6,RibFrac104,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac104-label.nii.gz,RibFrac104-label.nii.gz
7,RibFrac105,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac105-label.nii.gz,RibFrac105-label.nii.gz
8,RibFrac106,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac106-label.nii.gz,RibFrac106-label.nii.gz
9,RibFrac107,train_part1,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac107-label.nii.gz,RibFrac107-label.nii.gz



Index saved to:
/kaggle/working/ribfrac_metadata/ribfrac_label_file_index_500.csv

CELL #10 VALIDATION: PASS


## 11. Create patient → label mapping

In [12]:

# ============================================================
# CELL #11 — CREATE PATIENT-TO-LABEL MAPPING
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. Verify label index from Cell #10
# ------------------------------------------------------------

required_columns = [
    "patient_id",
    "source_partition",
    "archive_path",
    "label_member",
    "label_filename"
]

assert all(
    col in label_index_df.columns
    for col in required_columns
), "Required label index columns are missing."

assert len(label_index_df) == 500

assert not label_index_df["patient_id"].duplicated().any(), (
    "Duplicate patient IDs found."
)

# ------------------------------------------------------------
# 2. Initialize patient-to-label dictionaries
# ------------------------------------------------------------

# Compatibility with the original notebook:
# patient_id -> internal ZIP member filename
label_index = {}

# Complete mapping:
# patient_id -> source archive and annotation details
label_lookup = {}

# ------------------------------------------------------------
# 3. Build patient-to-label mapping
# ------------------------------------------------------------

for row in label_index_df.itertuples(index=False):

    patient_id = row.patient_id

    # Original mapping format
    label_index[patient_id] = row.label_member

    # Extended mapping for all three ZIP archives
    label_lookup[patient_id] = {
        "source_partition": row.source_partition,
        "archive_path": Path(row.archive_path),
        "label_member": row.label_member,
        "label_filename": row.label_filename
    }

# ------------------------------------------------------------
# 4. Validate mappings
# ------------------------------------------------------------

assert len(label_index) == 500
assert len(label_lookup) == 500

assert set(label_lookup.keys()) == set(patient_ids)

for patient_id, details in label_lookup.items():

    assert details["archive_path"].is_file(), (
        f"Archive missing for {patient_id}"
    )

    assert (
        patient_source_map[patient_id]
        == details["source_partition"]
    ), f"Source mismatch for {patient_id}"

    assert (
        label_index[patient_id]
        == details["label_member"]
    )

# ------------------------------------------------------------
# 5. Display examples
# ------------------------------------------------------------

print("=" * 60)
print("RIBFRAC PATIENT-TO-LABEL MAPPING")
print("=" * 60)

print("\nTotal patients indexed:", len(label_lookup))

print("\nPatients indexed by source:")

for source_name in RIBFRAC_SOURCES:

    count = sum(
        details["source_partition"] == source_name
        for details in label_lookup.values()
    )

    expected = RIBFRAC_SOURCES[
        source_name
    ]["expected_patients"]

    print(f"{source_name}: {count}")

    assert count == expected

# Example: original Part 1 patient
example_id = "RibFrac128"

if example_id in label_lookup:

    print(f"\nExample patient: {example_id}")

    for key, value in label_lookup[example_id].items():
        print(f"{key}: {value}")

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PATIENT-TO-LABEL MAPPING COMPLETED")
print("=" * 60)

print("Total indexed patients:", len(label_lookup))
print("All patient IDs matched successfully.")

print("\nCELL #11 VALIDATION: PASS")


RIBFRAC PATIENT-TO-LABEL MAPPING

Total patients indexed: 500

Patients indexed by source:
train_part1: 300
train_part2: 120
validation: 80

Example patient: RibFrac128
source_partition: train_part1
archive_path: /kaggle/working/ribfrac_label_archives/ribfrac-train-labels-1.zip
label_member: Part1/RibFrac128-label.nii.gz
label_filename: RibFrac128-label.nii.gz

PATIENT-TO-LABEL MAPPING COMPLETED
Total indexed patients: 500
All patient IDs matched successfully.

CELL #11 VALIDATION: PASS


## 12. Extract a patient's label

In [13]:

# ============================================================
# CELL #12 — EXTRACT A PATIENT'S RIBFRAC LABEL
# ============================================================

import zipfile
import shutil
from pathlib import Path, PurePosixPath

def get_label_path(patient_id):
    """
    Retrieve a patient's RibFrac annotation from the
    correct ZIP archive.

    Supports:
        - Training Part 1 (300 patients)
        - Training Part 2 (120 patients)
        - Validation (80 patients)

    Extracts only the requested annotation.

    Returns:
        Path to the local .nii.gz annotation file.
    """

    # --------------------------------------------------------
    # 1. Verify patient ID
    # --------------------------------------------------------

    patient_id = str(patient_id)

    if patient_id not in label_lookup:
        raise ValueError(
            f"No label mapping found for patient: {patient_id}"
        )

    # --------------------------------------------------------
    # 2. Retrieve annotation source details
    # --------------------------------------------------------

    details = label_lookup[patient_id]

    source_partition = details["source_partition"]
    archive_path = Path(details["archive_path"])
    member = details["label_member"]

    # --------------------------------------------------------
    # 3. Validate source archive
    # --------------------------------------------------------

    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Label archive not found: {archive_path}"
        )

    if source_partition != patient_source_map[patient_id]:
        raise ValueError(
            f"Source partition mismatch: {patient_id}"
        )

    # --------------------------------------------------------
    # 4. Determine safe local extraction path
    # --------------------------------------------------------

    relative_path = PurePosixPath(member)

    if (
        relative_path.is_absolute()
        or ".." in relative_path.parts
        or relative_path.name != f"{patient_id}-label.nii.gz"
    ):
        raise ValueError(
            f"Invalid ZIP member path: {member}"
        )

    # Preserve the archive's internal directory structure
    output_path = LABEL_DIR.joinpath(*relative_path.parts)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # 5. Extract only when necessary
    # --------------------------------------------------------

    with zipfile.ZipFile(archive_path, "r") as archive:

        file_info = archive.getinfo(member)

        # Reuse existing file if its size is correct
        if (
            output_path.is_file()
            and output_path.stat().st_size == file_info.file_size
        ):
            return output_path

        print(f"Extracting label for {patient_id}")
        print(f"Source: {source_partition}")

        temporary_path = output_path.with_name(
            output_path.name + ".part"
        )

        try:
            with archive.open(member, "r") as src:
                with open(temporary_path, "wb") as dst:
                    shutil.copyfileobj(src, dst)

            if temporary_path.stat().st_size != file_info.file_size:
                raise IOError(
                    f"Incomplete extraction for {patient_id}"
                )

            temporary_path.replace(output_path)

        except Exception:
            temporary_path.unlink(missing_ok=True)
            raise

    # --------------------------------------------------------
    # 6. Verify extracted file
    # --------------------------------------------------------

    assert output_path.is_file()
    assert output_path.stat().st_size > 0

    return output_path


print("Patient label extraction function configured.")
print("CELL #12 FUNCTION DEFINED: PASS")


Patient label extraction function configured.
CELL #12 FUNCTION DEFINED: PASS


## 13. Build the remote CT ZIP index

In [14]:

# ============================================================
# CELL #13 — CONFIGURE REMOTE RIBFRAC CT ZIP ACCESS
# ============================================================


# ------------------------------------------------------------
# 1. Configure CT archive URLs for all three sources
# ------------------------------------------------------------

CT_ZIP_URLS = {
    source_name: source_info["ct_url"]
    for source_name, source_info
    in RIBFRAC_SOURCES.items()
}

EXPECTED_CT_COUNTS = {
    source_name: source_info["expected_patients"]
    for source_name, source_info
    in RIBFRAC_SOURCES.items()
}

assert len(CT_ZIP_URLS) == 3
assert sum(EXPECTED_CT_COUNTS.values()) == 500

# ------------------------------------------------------------
# 2. Resolve the CT archive associated with a patient
# ------------------------------------------------------------

def get_patient_ct_source(patient_id):
    """
    Return the source partition for a RibFrac patient.
    """

    patient_id = str(patient_id)

    if patient_id not in patient_source_map:
        raise ValueError(
            f"Unknown RibFrac patient: {patient_id}"
        )

    source_name = patient_source_map[patient_id]

    if source_name not in CT_ZIP_URLS:
        raise ValueError(
            f"No CT archive configured for {source_name}"
        )

    return source_name


# ------------------------------------------------------------
# 3. Request bytes from a remote CT ZIP archive
# ------------------------------------------------------------


# ============================================================
# CELL #13 FIX — RETRY FAILED ZENODO RANGE REQUESTS
# ============================================================



def _request_ct_range(
    range_header,
    source_name,
    max_retries=6
):
    """
    Retrieve a byte range from a RibFrac ZIP archive.

    Automatically retries temporary HTTP failures:
    429, 500, 502, 503, 504.

    Returns:
        data, actual_start, actual_end
    """

    if source_name not in CT_ZIP_URLS:
        raise ValueError(
            f"Unknown CT source: {source_name}"
        )

    url = CT_ZIP_URLS[source_name]

    headers = {
        "Range": range_header,
        "Accept-Encoding": "identity"
    }

    retryable_status = {
        429, 500, 502, 503, 504
    }

    last_error = None

    for attempt in range(1, max_retries + 1):

        try:

            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 180),
                allow_redirects=True
            ) as response:

                status = response.status_code

                if status in retryable_status:
                    raise requests.HTTPError(
                        f"Temporary HTTP {status}"
                    )

                if status != 206:
                    raise RuntimeError(
                        f"{source_name}: Expected HTTP 206, "
                        f"received HTTP {status}. "
                        "The full ZIP will not be downloaded."
                    )

                content_range = response.headers.get(
                    "Content-Range", ""
                )

                match = re.fullmatch(
                    r"bytes (\d+)-(\d+)/(\d+|\*)",
                    content_range
                )

                if match is None:
                    raise RuntimeError(
                        "Invalid Content-Range: "
                        f"{content_range}"
                    )

                actual_start = int(match.group(1))
                actual_end = int(match.group(2))

                data = response.content

                expected_size = (
                    actual_end - actual_start + 1
                )

                if len(data) != expected_size:
                    raise requests.ConnectionError(
                        "Incomplete range response: "
                        f"{len(data)} / {expected_size} bytes"
                    )

                return (
                    data,
                    actual_start,
                    actual_end
                )

        except (
            requests.RequestException
        ) as error:

            last_error = error

            print(
                f"\nZenodo request failed "
                f"({attempt}/{max_retries}): {error}"
            )

            if attempt == max_retries:
                break

            delay = min(
                2 ** attempt,
                20
            )

            print(
                f"Retrying after {delay} seconds..."
            )

            time.sleep(delay)

    raise RuntimeError(
        f"{source_name}: Remote ZIP request failed "
        f"after {max_retries} attempts. "
        f"Range: {range_header}. "
        f"Last error: {last_error}"
    ) from last_error


print("HTTP range retry function configured.")
print("CELL #13 RETRY FIX: PASS")

# ------------------------------------------------------------
# 4. Download an exact byte interval
# ------------------------------------------------------------

def http_range(start, end, source_name="train_part1"):
    """
    Download byte interval [start, end] (inclusive).

    source_name:
        train_part1
        train_part2
        validation
    """

    start = int(start)
    end = int(end)

    if start < 0 or end < start:
        raise ValueError(
            f"Invalid byte range: {start}-{end}"
        )

    data, actual_start, actual_end = _request_ct_range(
        f"bytes={start}-{end}",
        source_name
    )

    if actual_start != start or actual_end != end:
        raise RuntimeError(
            f"{source_name}: Server returned unexpected "
            f"byte range {actual_start}-{actual_end}."
        )

    return data


# ------------------------------------------------------------
# 5. Download bytes from the end of a remote archive
# ------------------------------------------------------------

def http_suffix_range(
    nbytes=131072,
    source_name="train_part1"
):
    """
    Read the last nbytes of a remote ZIP archive.

    Used to locate ZIP64 End of Central Directory
    structures in Cell #14.
    """

    nbytes = int(nbytes)

    if nbytes <= 0:
        raise ValueError(
            "nbytes must be a positive integer."
        )

    data, _, _ = _request_ct_range(
        f"bytes=-{nbytes}",
        source_name
    )

    return data


# ------------------------------------------------------------
# 6. Initialize per-source CT indexes
# ------------------------------------------------------------

CT_ZIP_INDEXES = {}

# Preserve the original Part 1 URL alias
CT_ZIP_URL = CT_ZIP_URLS["train_part1"]

# ------------------------------------------------------------
# 7. Display configuration
# ------------------------------------------------------------

print("=" * 60)
print("RIBFRAC REMOTE CT ARCHIVE CONFIGURATION")
print("=" * 60)

for source_name, url in CT_ZIP_URLS.items():

    print(f"\nSource: {source_name}")
    print(
        "Expected CT examinations:",
        EXPECTED_CT_COUNTS[source_name]
    )
    print("Remote URL:", url)

print("\nTotal expected CT examinations: 500")

print("\nHTTP range functions configured.")
print("CELL #13 CONFIGURATION: PASS")


HTTP range retry function configured.
CELL #13 RETRY FIX: PASS
RIBFRAC REMOTE CT ARCHIVE CONFIGURATION

Source: train_part1
Expected CT examinations: 300
Remote URL: https://zenodo.org/records/3893508/files/ribfrac-train-images-1.zip?download=1

Source: train_part2
Expected CT examinations: 120
Remote URL: https://zenodo.org/records/3893498/files/ribfrac-train-images-2.zip?download=1

Source: validation
Expected CT examinations: 80
Remote URL: https://zenodo.org/records/3893496/files/ribfrac-val-images.zip?download=1

Total expected CT examinations: 500

HTTP range functions configured.
CELL #13 CONFIGURATION: PASS


In [16]:

# ============================================================
# TEST ZENODO RANGE ACCESS
# ============================================================

try:
    data = http_range(
        0,
        1023,
        source_name="train_part1"
    )

    print("Received:", len(data), "bytes")

    assert len(data) == 1024

    print("ZENODO CONNECTION TEST: PASS")

except Exception as error:
    print("ZENODO CONNECTION TEST: FAILED")
    print(type(error).__name__, error)


Received: 1024 bytes
ZENODO CONNECTION TEST: PASS


## 14. Read ZIP64 information

In [17]:

# ============================================================
# CELL #14 — READ RIBFRAC ZIP64 INFORMATION
# ============================================================

import struct

# ------------------------------------------------------------
# 1. Locate the ZIP64 End of Central Directory
# ------------------------------------------------------------

def find_zip64_eocd(
    source_name="train_part1",
    tail_size=131072
):
    """
    Locate the ZIP64 End of Central Directory (EOCD)
    record in a remote RibFrac CT ZIP archive.

    Parameters
    ----------
    source_name : str
        train_part1, train_part2, or validation

    tail_size : int
        Number of bytes requested from archive end.

    Returns
    -------
    int
        Absolute byte offset of the ZIP64 EOCD record.
    """

    if source_name not in CT_ZIP_URLS:
        raise ValueError(
            f"Unknown CT source: {source_name}"
        )

    # --------------------------------------------------------
    # 2. Read the end of the remote ZIP archive
    # --------------------------------------------------------

    data = http_suffix_range(
        nbytes=tail_size,
        source_name=source_name
    )

    print(f"\nSource: {source_name}")
    print("Tail bytes retrieved:", len(data))

    # ZIP signatures
    eocd_signature = b"PK\x05\x06"
    locator_signature = b"PK\x06\x07"
    zip64_signature = b"PK\x06\x06"

    # --------------------------------------------------------
    # 3. Find a valid standard ZIP EOCD record
    # --------------------------------------------------------

    eocd_pos = data.rfind(eocd_signature)

    while eocd_pos >= 0:

        if eocd_pos + 22 <= len(data):

            comment_length = struct.unpack_from(
                "<H",
                data,
                eocd_pos + 20
            )[0]

            # Valid EOCD should finish at archive end
            if (
                eocd_pos + 22 + comment_length
                == len(data)
            ):
                break

        eocd_pos = data.rfind(
            eocd_signature,
            0,
            eocd_pos
        )

    if eocd_pos < 0:
        raise RuntimeError(
            f"{source_name}: Valid ZIP EOCD not found."
        )

    # --------------------------------------------------------
    # 4. Locate the ZIP64 EOCD locator
    # --------------------------------------------------------

    # The ZIP64 locator is 20 bytes immediately
    # before the standard ZIP EOCD record.

    locator_pos = eocd_pos - 20

    if (
        locator_pos < 0
        or data[locator_pos:locator_pos + 4]
        != locator_signature
    ):
        raise RuntimeError(
            f"{source_name}: ZIP64 locator not found. "
            "The archive may use standard ZIP rather "
            "than ZIP64."
        )

    # --------------------------------------------------------
    # 5. Read ZIP64 locator fields
    # --------------------------------------------------------

    disk_number = struct.unpack_from(
        "<I",
        data,
        locator_pos + 4
    )[0]

    zip64_eocd_offset = struct.unpack_from(
        "<Q",
        data,
        locator_pos + 8
    )[0]

    total_disks = struct.unpack_from(
        "<I",
        data,
        locator_pos + 16
    )[0]

    if disk_number != 0 or total_disks != 1:
        raise RuntimeError(
            f"{source_name}: Multi-disk ZIP archives "
            "are not supported."
        )

    # --------------------------------------------------------
    # 6. Verify the ZIP64 EOCD record
    # --------------------------------------------------------

    zip64_header = http_range(
        zip64_eocd_offset,
        zip64_eocd_offset + 55,
        source_name=source_name
    )

    if zip64_header[:4] != zip64_signature:
        raise RuntimeError(
            f"{source_name}: Invalid ZIP64 EOCD signature."
        )

    record_size = struct.unpack_from(
        "<Q",
        zip64_header,
        4
    )[0]

    if record_size < 44:
        raise RuntimeError(
            f"{source_name}: Invalid ZIP64 record size."
        )

    # --------------------------------------------------------
    # 7. Extract central directory information
    # --------------------------------------------------------

    total_entries = struct.unpack_from(
        "<Q",
        zip64_header,
        32
    )[0]

    central_dir_size = struct.unpack_from(
        "<Q",
        zip64_header,
        40
    )[0]

    central_dir_offset = struct.unpack_from(
        "<Q",
        zip64_header,
        48
    )[0]

    if (
        central_dir_offset + central_dir_size
        > zip64_eocd_offset
    ):
        raise RuntimeError(
            f"{source_name}: Invalid central directory "
            "offset or size."
        )

    print("ZIP64 EOCD offset:", zip64_eocd_offset)
    print("Total ZIP entries:", total_entries)
    print("Central directory size:", central_dir_size)
    print("Central directory offset:", central_dir_offset)
    print("ZIP64 validation: PASS")

    return zip64_eocd_offset


# ------------------------------------------------------------
# 8. Read ZIP64 information for all three sources
# ------------------------------------------------------------

zip64_eocd_offsets = {}

for source_name in CT_ZIP_URLS:

    offset = find_zip64_eocd(
        source_name=source_name
    )

    zip64_eocd_offsets[source_name] = offset


# ------------------------------------------------------------
# 9. Final verification
# ------------------------------------------------------------

assert len(zip64_eocd_offsets) == 3

assert all(
    isinstance(offset, int) and offset > 0
    for offset in zip64_eocd_offsets.values()
)

print("\n" + "=" * 60)
print("RIBFRAC ZIP64 INFORMATION")
print("=" * 60)

for source_name, offset in zip64_eocd_offsets.items():

    print(
        f"{source_name}: ZIP64 EOCD offset = {offset}"
    )

print("\nCELL #14 VALIDATION: PASS")



Source: train_part1
Tail bytes retrieved: 131072
ZIP64 EOCD offset: 36605667070
Total ZIP entries: 301
Central directory size: 32848
Central directory offset: 36605634222
ZIP64 validation: PASS

Source: train_part2
Tail bytes retrieved: 131072
ZIP64 EOCD offset: 14562971375
Total ZIP entries: 121
Central directory size: 12976
Central directory offset: 14562958399
ZIP64 validation: PASS

Source: validation
Tail bytes retrieved: 131072
ZIP64 EOCD offset: 8729000015
Total ZIP entries: 81
Central directory size: 9517
Central directory offset: 8728990498
ZIP64 validation: PASS

RIBFRAC ZIP64 INFORMATION
train_part1: ZIP64 EOCD offset = 36605667070
train_part2: ZIP64 EOCD offset = 14562971375
validation: ZIP64 EOCD offset = 8729000015

CELL #14 VALIDATION: PASS


## 15. Parse the central directory

In [18]:

# ============================================================
# CELL #15 — PARSE REMOTE RIBFRAC CT CENTRAL DIRECTORIES
# ============================================================

import struct
import pandas as pd
from pathlib import PurePosixPath

# ------------------------------------------------------------
# 1. Read ZIP64 extended fields from a central directory entry
# ------------------------------------------------------------

def parse_zip64_extra(
    extra_data,
    uncompressed_size,
    compressed_size,
    local_header_offset,
    disk_start
):
    """
    Resolve 64-bit ZIP values when standard ZIP fields
    contain ZIP64 placeholder values.
    """

    need_uncompressed = uncompressed_size == 0xFFFFFFFF
    need_compressed = compressed_size == 0xFFFFFFFF
    need_offset = local_header_offset == 0xFFFFFFFF
    need_disk = disk_start == 0xFFFF

    if not any([
        need_uncompressed,
        need_compressed,
        need_offset,
        need_disk
    ]):
        return (
            uncompressed_size,
            compressed_size,
            local_header_offset,
            disk_start
        )

    # Find the ZIP64 extended information extra field (0x0001)
    pos = 0
    zip64_data = None

    while pos + 4 <= len(extra_data):

        field_id, field_size = struct.unpack_from(
            "<HH", extra_data, pos
        )

        field_end = pos + 4 + field_size

        if field_end > len(extra_data):
            raise RuntimeError(
                "Malformed ZIP extra field."
            )

        if field_id == 0x0001:
            zip64_data = extra_data[pos + 4:field_end]
            break

        pos = field_end

    if zip64_data is None:
        raise RuntimeError(
            "Required ZIP64 extra field not found."
        )

    offset = 0

    def read_value(fmt):
        nonlocal offset

        size = struct.calcsize(fmt)

        if offset + size > len(zip64_data):
            raise RuntimeError(
                "Incomplete ZIP64 extended field."
            )

        value = struct.unpack_from(
            fmt, zip64_data, offset
        )[0]

        offset += size
        return value

    # ZIP64 fields appear in this specified order.
    if need_uncompressed:
        uncompressed_size = read_value("<Q")

    if need_compressed:
        compressed_size = read_value("<Q")

    if need_offset:
        local_header_offset = read_value("<Q")

    if need_disk:
        disk_start = read_value("<I")

    return (
        uncompressed_size,
        compressed_size,
        local_header_offset,
        disk_start
    )


# ------------------------------------------------------------
# 2. Build a CT index for one source archive
# ------------------------------------------------------------

def build_ct_zip_index(source_name="train_part1"):
    """
    Read the central directory of one remote RibFrac
    CT archive and construct a file index.

    Does not download the actual CT volumes.
    """

    if source_name not in CT_ZIP_URLS:
        raise ValueError(
            f"Unknown CT source: {source_name}"
        )

    print("\n" + "=" * 65)
    print("BUILDING CT ZIP INDEX:", source_name)
    print("=" * 65)

    # --------------------------------------------------------
    # 3. Retrieve ZIP64 EOCD offset from Cell #14
    # --------------------------------------------------------

    zip64_eocd_offset = zip64_eocd_offsets[source_name]

    eocd = http_range(
        zip64_eocd_offset,
        zip64_eocd_offset + 55,
        source_name=source_name
    )

    if eocd[:4] != b"PK\x06\x06":
        raise RuntimeError(
            f"{source_name}: Invalid ZIP64 EOCD signature."
        )

    total_entries = struct.unpack_from(
        "<Q", eocd, 32
    )[0]

    central_dir_size = struct.unpack_from(
        "<Q", eocd, 40
    )[0]

    central_dir_offset = struct.unpack_from(
        "<Q", eocd, 48
    )[0]

    print("Total ZIP entries:", total_entries)
    print("Central directory size:", central_dir_size)
    print("Central directory offset:", central_dir_offset)

    if central_dir_size <= 0:
        raise RuntimeError(
            f"{source_name}: Empty central directory."
        )

    # --------------------------------------------------------
    # 4. Download only the central directory bytes
    # --------------------------------------------------------

    central_data = http_range(
        central_dir_offset,
        central_dir_offset + central_dir_size - 1,
        source_name=source_name
    )

    print(
        "Central directory downloaded:",
        len(central_data), "bytes"
    )

    # --------------------------------------------------------
    # 5. Parse the central directory entries
    # --------------------------------------------------------

    entries = []
    pos = 0

    while pos < len(central_data):

        signature = central_data[pos:pos + 4]

        if signature != b"PK\x01\x02":

            # An optional central-directory digital signature
            # may appear after the file entries.
            if (
                len(entries) == total_entries
                and signature == b"PK\x05\x05"
            ):
                break

            raise RuntimeError(
                f"{source_name}: Invalid central directory "
                f"signature at offset {pos}."
            )

        if pos + 46 > len(central_data):
            raise RuntimeError(
                "Incomplete central directory entry."
            )

        flags = struct.unpack_from(
            "<H", central_data, pos + 8
        )[0]

        compression_method = struct.unpack_from(
            "<H", central_data, pos + 10
        )[0]

        crc32 = struct.unpack_from(
            "<I", central_data, pos + 16
        )[0]

        compressed_size = struct.unpack_from(
            "<I", central_data, pos + 20
        )[0]

        uncompressed_size = struct.unpack_from(
            "<I", central_data, pos + 24
        )[0]

        filename_length = struct.unpack_from(
            "<H", central_data, pos + 28
        )[0]

        extra_length = struct.unpack_from(
            "<H", central_data, pos + 30
        )[0]

        comment_length = struct.unpack_from(
            "<H", central_data, pos + 32
        )[0]

        disk_start = struct.unpack_from(
            "<H", central_data, pos + 34
        )[0]

        local_header_offset = struct.unpack_from(
            "<I", central_data, pos + 42
        )[0]

        filename_start = pos + 46
        filename_end = filename_start + filename_length

        extra_end = filename_end + extra_length
        entry_end = extra_end + comment_length

        if entry_end > len(central_data):
            raise RuntimeError(
                "Central directory entry exceeds available data."
            )

        filename_bytes = central_data[
            filename_start:filename_end
        ]

        encoding = (
            "utf-8" if flags & 0x800 else "cp437"
        )

        filename = filename_bytes.decode(encoding)

        extra_data = central_data[
            filename_end:extra_end
        ]

        # ----------------------------------------------------
        # 6. Resolve ZIP64 file sizes and offsets
        # ----------------------------------------------------

        (
            uncompressed_size,
            compressed_size,
            local_header_offset,
            disk_start
        ) = parse_zip64_extra(
            extra_data,
            uncompressed_size,
            compressed_size,
            local_header_offset,
            disk_start
        )

        if disk_start != 0:
            raise RuntimeError(
                "Multi-disk ZIP files are not supported."
            )

        # Retain actual CT image files, excluding folders.
        if filename.endswith("-image.nii.gz"):

            if flags & 0x1:
                raise RuntimeError(
                    f"Encrypted CT entry: {filename}"
                )

            if compression_method not in (0, 8):
                raise RuntimeError(
                    f"Unsupported ZIP compression method "
                    f"{compression_method}: {filename}"
                )

            patient_id = (
                PurePosixPath(filename).name
                .removesuffix("-image.nii.gz")
            )

            entries.append({
                "patient_id": patient_id,
                "source_partition": source_name,
                "filename": filename,
                "compressed_size": int(compressed_size),
                "uncompressed_size": int(uncompressed_size),
                "local_header_offset": int(local_header_offset),
                "compression_method": int(compression_method),
                "crc32": int(crc32)
            })

        # Advance to next central directory entry
        pos = entry_end

        # Count all entries, including directories,
        # through a separate counter below.

    # --------------------------------------------------------
    # 7. Construct source DataFrame
    # --------------------------------------------------------

    source_df = pd.DataFrame(entries)

    expected = EXPECTED_CT_COUNTS[source_name]

    print("CT image entries found:", len(source_df))
    print("Expected CT images:", expected)

    if len(source_df) != expected:
        raise RuntimeError(
            f"{source_name}: Expected {expected} CT images, "
            f"found {len(source_df)}."
        )

    if source_df["patient_id"].duplicated().any():
        raise RuntimeError(
            f"{source_name}: Duplicate CT patient IDs."
        )

    # Compare against the metadata for this source
    expected_patients = set(
        patient_registry.loc[
            patient_registry["source_partition"] == source_name,
            "public_id"
        ]
    )

    indexed_patients = set(source_df["patient_id"])

    if indexed_patients != expected_patients:
        raise RuntimeError(
            f"{source_name}: CT/metadata mismatch.\n"
            f"Missing CTs: "
            f"{sorted(expected_patients - indexed_patients)}\n"
            f"Unexpected CTs: "
            f"{sorted(indexed_patients - expected_patients)}"
        )

    print("CT/metadata matching: PASS")
    print("Source CT indexing: PASS")

    return source_df


# ------------------------------------------------------------
# 8. Build indexes for all three RibFrac CT archives
# ------------------------------------------------------------

CT_ZIP_INDEXES = {}

for source_name in CT_ZIP_URLS:

    CT_ZIP_INDEXES[source_name] = build_ct_zip_index(
        source_name
    )


# ------------------------------------------------------------
# 9. Combine the CT indexes
# ------------------------------------------------------------

ct_zip_index = pd.concat(
    CT_ZIP_INDEXES.values(),
    ignore_index=True
)

ct_zip_index = (
    ct_zip_index
    .sort_values(["source_partition", "patient_id"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 10. Validate the complete 500-patient CT index
# ------------------------------------------------------------

assert len(ct_zip_index) == 500, (
    f"Expected 500 CT files, found {len(ct_zip_index)}"
)

assert ct_zip_index["patient_id"].nunique() == 500

assert not ct_zip_index["patient_id"].duplicated().any()

assert set(ct_zip_index["patient_id"]) == set(patient_ids)

for row in ct_zip_index.itertuples(index=False):

    assert (
        patient_source_map[row.patient_id]
        == row.source_partition
    ), f"Source mismatch: {row.patient_id}"

    assert row.compressed_size > 0
    assert row.uncompressed_size > 0
    assert row.local_header_offset >= 0


# ------------------------------------------------------------
# 11. Save consolidated CT index
# ------------------------------------------------------------

CT_INDEX_PATH = (
    METADATA_DIR / "ribfrac_ct_file_index_500.csv"
)

ct_zip_index.to_csv(
    CT_INDEX_PATH,
    index=False
)


# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RIBFRAC CT CENTRAL DIRECTORY INDEX COMPLETED")
print("=" * 65)

print("\nCT image files by source:")

print(
    ct_zip_index["source_partition"]
    .value_counts()
    .sort_index()
)

print("\nTotal CT files:", len(ct_zip_index))
print("Unique patients:", ct_zip_index["patient_id"].nunique())

print("\nFirst 10 indexed CT files:")
display(ct_zip_index.head(10))

print("\nIndex saved to:")
print(CT_INDEX_PATH)

print("\nCELL #15 VALIDATION: PASS")



BUILDING CT ZIP INDEX: train_part1
Total ZIP entries: 301
Central directory size: 32848
Central directory offset: 36605634222
Central directory downloaded: 32848 bytes
CT image entries found: 300
Expected CT images: 300
CT/metadata matching: PASS
Source CT indexing: PASS

BUILDING CT ZIP INDEX: train_part2
Total ZIP entries: 121
Central directory size: 12976
Central directory offset: 14562958399
Central directory downloaded: 12976 bytes
CT image entries found: 120
Expected CT images: 120
CT/metadata matching: PASS
Source CT indexing: PASS

BUILDING CT ZIP INDEX: validation
Total ZIP entries: 81
Central directory size: 9517
Central directory offset: 8728990498
Central directory downloaded: 9517 bytes
CT image entries found: 80
Expected CT images: 80
CT/metadata matching: PASS
Source CT indexing: PASS

RIBFRAC CT CENTRAL DIRECTORY INDEX COMPLETED

CT image files by source:
source_partition
train_part1    300
train_part2    120
validation      80
Name: count, dtype: int64

Total CT files

,patient_id,source_partition,filename,compressed_size,uncompressed_size,local_header_offset,compression_method,crc32
0,RibFrac1,train_part1,Part1/RibFrac1-image.nii.gz,109443421,110682920,15395682268,8,3347870264
1,RibFrac10,train_part1,Part1/RibFrac10-image.nii.gz,118808918,120127942,14391836808,8,1552896568
2,RibFrac100,train_part1,Part1/RibFrac100-image.nii.gz,110652903,111890711,3435146294,8,251323667
3,RibFrac101,train_part1,Part1/RibFrac101-image.nii.gz,119710494,120978682,3315435713,8,3487095889
4,RibFrac102,train_part1,Part1/RibFrac102-image.nii.gz,141460917,141455047,3173974709,8,2274132388
5,RibFrac103,train_part1,Part1/RibFrac103-image.nii.gz,118313848,119604634,3055660774,8,2164004797
6,RibFrac104,train_part1,Part1/RibFrac104-image.nii.gz,107626450,108886518,2948034237,8,2024215928
7,RibFrac105,train_part1,Part1/RibFrac105-image.nii.gz,117939886,117933519,2830094264,8,3466363441
8,RibFrac106,train_part1,Part1/RibFrac106-image.nii.gz,138849679,140334227,2691244498,8,3066793188
9,RibFrac107,train_part1,Part1/RibFrac107-image.nii.gz,123283616,124715514,2567960795,8,7905595



Index saved to:
/kaggle/working/ribfrac_metadata/ribfrac_ct_file_index_500.csv

CELL #15 VALIDATION: PASS


## 16. — Get a Patient's CT Entry

In [19]:

# ============================================================
# CELL #16 — GET A PATIENT'S CT ENTRY
# ============================================================

import pandas as pd

def get_ct_entry(patient_id):
    """
    Locate a patient's CT image in the unified RibFrac
    remote ZIP index.

    Supports:
        - Training Part 1 (300 patients)
        - Training Part 2 (120 patients)
        - Validation (80 patients)

    Returns
    -------
    pandas.Series
        The matching CT entry with its source partition,
        ZIP member filename, file sizes, compression
        method, CRC32, and local header offset.
    """

    # --------------------------------------------------------
    # 1. Verify patient identifier
    # --------------------------------------------------------

    patient_id = str(patient_id)

    if patient_id not in patient_source_map:
        raise ValueError(
            f"Unknown RibFrac patient: {patient_id}"
        )

    # --------------------------------------------------------
    # 2. Determine patient's original source partition
    # --------------------------------------------------------

    source_name = patient_source_map[patient_id]

    if source_name not in CT_ZIP_URLS:
        raise ValueError(
            f"No CT archive configured for {source_name}"
        )

    # --------------------------------------------------------
    # 3. Find patient in the combined CT index
    # --------------------------------------------------------

    matches = ct_zip_index.loc[
        (ct_zip_index["patient_id"] == patient_id) &
        (ct_zip_index["source_partition"] == source_name)
    ]

    # --------------------------------------------------------
    # 4. Validate the matching entry
    # --------------------------------------------------------

    if matches.empty:
        raise FileNotFoundError(
            f"CT entry not found for {patient_id} "
            f"in source {source_name}"
        )

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one CT entry for {patient_id}, "
            f"found {len(matches)}"
        )

    entry = matches.iloc[0].copy()

    expected_filename = f"{patient_id}-image.nii.gz"

    if not str(entry["filename"]).endswith(
        "/" + expected_filename
    ) and str(entry["filename"]) != expected_filename:
        raise RuntimeError(
            f"Unexpected CT filename: {entry['filename']}"
        )

    # Verify ZIP metadata
    assert int(entry["compressed_size"]) > 0
    assert int(entry["uncompressed_size"]) > 0
    assert int(entry["local_header_offset"]) >= 0

    assert int(entry["compression_method"]) in (0, 8)

    return entry


# ============================================================
# TEST CT ENTRY LOOKUP FOR ALL THREE SOURCES
# ============================================================

print("=" * 60)
print("RIBFRAC PATIENT CT ENTRY LOOKUP")
print("=" * 60)

for source_name in RIBFRAC_SOURCES:

    source_patients = patient_registry.loc[
        patient_registry["source_partition"] == source_name,
        "public_id"
    ].tolist()

    assert source_patients, (
        f"No patients found for {source_name}"
    )

    patient_id = source_patients[0]

    entry = get_ct_entry(patient_id)

    print(f"\nSource: {source_name}")
    print(f"Patient: {patient_id}")
    print(f"CT filename: {entry['filename']}")
    print(
        "Compressed size:",
        f"{int(entry['compressed_size']) / 1024**2:.2f} MB"
    )
    print(
        "Uncompressed size:",
        f"{int(entry['uncompressed_size']) / 1024**2:.2f} MB"
    )
    print(
        "Local header offset:",
        int(entry["local_header_offset"])
    )
    print(
        "Compression method:",
        int(entry["compression_method"])
    )

    assert entry["source_partition"] == source_name

    print("CT entry lookup: PASS")

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

assert len(ct_zip_index) == 500
assert ct_zip_index["patient_id"].nunique() == 500

print("\n" + "=" * 60)
print("CELL #16 VALIDATION: PASS")
print("=" * 60)


RIBFRAC PATIENT CT ENTRY LOOKUP

Source: train_part1
Patient: RibFrac1
CT filename: Part1/RibFrac1-image.nii.gz
Compressed size: 104.37 MB
Uncompressed size: 105.56 MB
Local header offset: 15395682268
Compression method: 8
CT entry lookup: PASS

Source: train_part2
Patient: RibFrac301
CT filename: Part2/RibFrac301-image.nii.gz
Compressed size: 125.86 MB
Uncompressed size: 125.84 MB
Local header offset: 14430987908
Compression method: 8
CT entry lookup: PASS

Source: validation
Patient: RibFrac421
CT filename: ribfrac-val-images/RibFrac421-image.nii.gz
Compressed size: 102.26 MB
Uncompressed size: 103.43 MB
Local header offset: 8621760992
Compression method: 8
CT entry lookup: PASS

CELL #16 VALIDATION: PASS


## 17. Remote CT downloader

In [20]:
#17. Remote CT downloader

# ============================================================
# CELL #17 — REMOTE RIBFRAC CT DOWNLOADER
# Supports all 500 publicly annotated examinations
# ============================================================

import zlib
import struct
from pathlib import Path

def download_ribfrac_image(
    patient_id,
    chunk_size=1024 * 1024
):
    """
    Download one RibFrac CT image from its remote
    ZIP archive using HTTP byte-range requests.

    Supports:
        train_part1
        train_part2
        validation

    Features:
        - On-demand CT retrieval
        - Incremental disk writing
        - ZIP64-compatible offsets
        - DEFLATE and STORED ZIP entries
        - Size and CRC32 validation
        - Reuse of verified local files

    Returns
    -------
    pathlib.Path
        Path to the extracted .nii.gz CT image.
    """

    # --------------------------------------------------------
    # 1. Validate patient and configuration
    # --------------------------------------------------------

    patient_id = str(patient_id)
    chunk_size = int(chunk_size)

    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")

    entry = get_ct_entry(patient_id)

    source_name = str(entry["source_partition"])

    if source_name != patient_source_map[patient_id]:
        raise RuntimeError(
            f"CT source mismatch for {patient_id}"
        )

    # --------------------------------------------------------
    # 2. Retrieve ZIP central-directory metadata
    # --------------------------------------------------------

    local_header_offset = int(
        entry["local_header_offset"]
    )

    compressed_size = int(
        entry["compressed_size"]
    )

    uncompressed_size = int(
        entry["uncompressed_size"]
    )

    compression_method = int(
        entry["compression_method"]
    )

    expected_crc32 = int(
        entry["crc32"]
    ) & 0xFFFFFFFF

    expected_member = str(entry["filename"])

    # --------------------------------------------------------
    # 3. Configure local output
    # --------------------------------------------------------

    CT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    output_path = (
        CT_DIR / f"{patient_id}-image.nii.gz"
    )

    temporary_path = (
        CT_DIR / f"{patient_id}-image.nii.gz.part"
    )

    # --------------------------------------------------------
    # 4. Check integrity of existing local file
    # --------------------------------------------------------

    def verify_existing_file(path):

        if not path.is_file():
            return False

        if path.stat().st_size != uncompressed_size:
            return False

        crc = 0

        with open(path, "rb") as file:

            while True:

                block = file.read(1024 * 1024)

                if not block:
                    break

                crc = zlib.crc32(block, crc)

        return (
            (crc & 0xFFFFFFFF)
            == expected_crc32
        )

    if verify_existing_file(output_path):

        print("Verified existing CT:", patient_id)
        print("Path:", output_path)

        return output_path

    # --------------------------------------------------------
    # 5. Read local ZIP member header
    # --------------------------------------------------------

    local_header = http_range(
        local_header_offset,
        local_header_offset + 29,
        source_name=source_name
    )

    if local_header[:4] != b"PK\x03\x04":

        raise RuntimeError(
            f"{patient_id}: Invalid local ZIP header."
        )

    flags = struct.unpack_from(
        "<H", local_header, 6
    )[0]

    local_method = struct.unpack_from(
        "<H", local_header, 8
    )[0]

    filename_length = struct.unpack_from(
        "<H", local_header, 26
    )[0]

    extra_length = struct.unpack_from(
        "<H", local_header, 28
    )[0]

    if flags & 0x1:
        raise RuntimeError(
            "Encrypted ZIP members are not supported."
        )

    if local_method != compression_method:
        raise RuntimeError(
            "ZIP compression-method mismatch."
        )

    if compression_method not in (0, 8):
        raise RuntimeError(
            f"Unsupported ZIP method: {compression_method}"
        )

    # --------------------------------------------------------
    # 6. Validate local member filename
    # --------------------------------------------------------

    name_start = local_header_offset + 30

    if filename_length <= 0:
        raise RuntimeError("Empty ZIP member filename.")

    name_bytes = http_range(
        name_start,
        name_start + filename_length - 1,
        source_name=source_name
    )

    encoding = (
        "utf-8" if flags & 0x800 else "cp437"
    )

    local_filename = name_bytes.decode(encoding)

    if local_filename != expected_member:
        raise RuntimeError(
            f"ZIP filename mismatch for {patient_id}."
        )

    # --------------------------------------------------------
    # 7. Determine compressed data position
    # --------------------------------------------------------

    data_start = (
        local_header_offset
        + 30
        + filename_length
        + extra_length
    )

    data_end = (
        data_start
        + compressed_size
        - 1
    )

    print("\n" + "=" * 60)
    print("DOWNLOADING RIBFRAC CT")
    print("=" * 60)

    print("Patient:", patient_id)
    print("Source:", source_name)
    print("ZIP member:", expected_member)

    print(
        "Compressed size:",
        f"{compressed_size / 1024**2:.2f} MB"
    )

    print(
        "Extracted size:",
        f"{uncompressed_size / 1024**2:.2f} MB"
    )

    print("Compression method:", compression_method)
    print("Data start:", data_start)
    print("Data end:", data_end)

    # --------------------------------------------------------
    # 8. Stream and extract the ZIP member
    # --------------------------------------------------------

    downloaded = 0
    written = 0
    crc = 0

    decompressor = (
        zlib.decompressobj(-zlib.MAX_WBITS)
        if compression_method == 8
        else None
    )

    # Keep decompressed output blocks bounded.
    output_block_size = 1024 * 1024

    def write_piece(file, piece):
        nonlocal written, crc

        if piece:
            file.write(piece)
            written += len(piece)
            crc = zlib.crc32(piece, crc)

    # Remove any stale partial file
    temporary_path.unlink(missing_ok=True)

    try:

        with open(temporary_path, "wb") as output_file:

            last_report = 0

            while downloaded < compressed_size:

                remaining = compressed_size - downloaded

                request_size = min(
                    chunk_size,
                    remaining
                )

                start = data_start + downloaded

                end = start + request_size - 1

                # Read only the requested bytes
                compressed_chunk = http_range(
                    start,
                    end,
                    source_name=source_name
                )

                if len(compressed_chunk) != request_size:
                    raise IOError(
                        "Incomplete CT download chunk."
                    )

                downloaded += len(compressed_chunk)

                if compression_method == 0:

                    # Stored ZIP member: no DEFLATE
                    write_piece(
                        output_file,
                        compressed_chunk
                    )

                else:

                    # Incrementally decompress raw DEFLATE
                    pending = compressed_chunk

                    while pending:

                        previous_size = len(pending)

                        piece = decompressor.decompress(
                            pending,
                            output_block_size
                        )

                        write_piece(output_file, piece)

                        pending = decompressor.unconsumed_tail

                        if (
                            not piece
                            and len(pending) >= previous_size
                        ):
                            raise RuntimeError(
                                "DEFLATE decompression stalled."
                            )

                    # Drain any internally buffered output
                    while True:

                        piece = decompressor.decompress(
                            b"",
                            output_block_size
                        )

                        if not piece:
                            break

                        write_piece(output_file, piece)

                # Report progress approximately every 25 MB
                if (
                    downloaded - last_report
                    >= 25 * 1024 * 1024
                    or downloaded == compressed_size
                ):

                    print(
                        f"Downloaded: "
                        f"{downloaded / 1024**2:.2f} / "
                        f"{compressed_size / 1024**2:.2f} MB"
                    )

                    last_report = downloaded

            # Finalize DEFLATE stream
            if decompressor is not None:

                final_piece = decompressor.flush()

                write_piece(
                    output_file,
                    final_piece
                )

                if not decompressor.eof:
                    raise RuntimeError(
                        "Incomplete DEFLATE stream."
                    )

                if decompressor.unused_data:
                    raise RuntimeError(
                        "Unexpected data after DEFLATE stream."
                    )

        # ----------------------------------------------------
        # 9. Validate extracted CT file
        # ----------------------------------------------------

        if downloaded != compressed_size:
            raise RuntimeError(
                "Compressed download size mismatch."
            )

        if written != uncompressed_size:
            raise RuntimeError(
                f"Extracted size mismatch: "
                f"{written} != {uncompressed_size}"
            )

        actual_crc32 = crc & 0xFFFFFFFF

        if actual_crc32 != expected_crc32:
            raise RuntimeError(
                f"CRC32 mismatch for {patient_id}: "
                f"{actual_crc32:08x} != "
                f"{expected_crc32:08x}"
            )

        # Save only after all checks have passed
        temporary_path.replace(output_path)

    except Exception:

        temporary_path.unlink(missing_ok=True)

        raise

    # --------------------------------------------------------
    # 10. Final verification
    # --------------------------------------------------------

    print("\nCT DOWNLOAD COMPLETED")

    print(
        "Extracted size:",
        f"{written / 1024**2:.2f} MB"
    )

    print(
        "CRC32:",
        f"{actual_crc32:08x}"
    )

    print("Saved:", output_path)
    print("CT ZIP extraction validation: PASS")

    return output_path


print("Remote CT downloader configured.")
print("CELL #17 FUNCTION DEFINED: PASS")


Remote CT downloader configured.
CELL #17 FUNCTION DEFINED: PASS


In [21]:

# ============================================================
# TEST CELL #17 — CT RETRIEVAL
# ============================================================

import nibabel as nib
from pathlib import Path

patient_id = "RibFrac128"

# Download or reuse verified CT
ct_path = download_ribfrac_image(patient_id)

# Verify file
assert ct_path.is_file()

# Read NIfTI header without loading the full CT array
ct_img = nib.load(str(ct_path))

print("\n" + "=" * 60)
print("RIBFRAC CT TEST")
print("=" * 60)

print("Patient:", patient_id)
print("CT path:", ct_path)
print("CT shape:", ct_img.shape)
print(
    "Voxel spacing:",
    ct_img.header.get_zooms()[:3]
)

print("Data type:", ct_img.get_data_dtype())

assert len(ct_img.shape) == 3

print("\nCELL #17 RETRIEVAL TEST: PASS")



DOWNLOADING RIBFRAC CT
Patient: RibFrac128
Source: train_part1
ZIP member: Part1/RibFrac128-image.nii.gz
Compressed size: 105.66 MB
Extracted size: 106.84 MB
Compression method: 8
Data start: 151
Data end: 110796638
Downloaded: 25.00 / 105.66 MB
Downloaded: 50.00 / 105.66 MB
Downloaded: 75.00 / 105.66 MB
Downloaded: 100.00 / 105.66 MB
Downloaded: 105.66 / 105.66 MB

CT DOWNLOAD COMPLETED
Extracted size: 106.84 MB
CRC32: 9c38b88d
Saved: /kaggle/working/ribfrac_images/RibFrac128-image.nii.gz
CT ZIP extraction validation: PASS

RIBFRAC CT TEST
Patient: RibFrac128
CT path: /kaggle/working/ribfrac_images/RibFrac128-image.nii.gz
CT shape: (512, 512, 337)
Voxel spacing: (np.float32(0.771484), np.float32(0.771484), np.float32(1.25))
Data type: int32

CELL #17 RETRIEVAL TEST: PASS


## 18. Test automatic CT retrieval

In [23]:

# ============================================================
# CELL #18 — TEST AUTOMATIC RIBFRAC CT RETRIEVAL
# ============================================================

import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path

# ------------------------------------------------------------
# 1. Verify required data structures
# ------------------------------------------------------------

assert len(patient_source_map) == 500
assert len(CT_ZIP_INDEXES) == 3
assert len(ct_zip_index) == 500

# ------------------------------------------------------------
# 2. Select one test patient from each source
# ------------------------------------------------------------

test_patients = {}

for source_name in RIBFRAC_SOURCES:

    source_patients = sorted(
        patient_registry.loc[
            patient_registry["source_partition"]
            == source_name,
            "public_id"
        ].tolist()
    )

    assert source_patients, (
        f"No patients found in {source_name}"
    )

    # Preserve original RibFrac128 test
    if (
        source_name == "train_part1"
        and "RibFrac128" in source_patients
    ):
        patient_id = "RibFrac128"
    else:
        patient_id = source_patients[0]

    test_patients[source_name] = patient_id

print("=" * 65)
print("RIBFRAC CT RETRIEVAL TEST CONFIGURATION")
print("=" * 65)

for source_name, patient_id in test_patients.items():
    print(f"{source_name}: {patient_id}")

# ------------------------------------------------------------
# 3. Download and validate one CT from each source
# ------------------------------------------------------------

ct_test_results = []

for source_name, patient_id in test_patients.items():

    print("\n" + "=" * 65)
    print(f"TESTING SOURCE: {source_name}")
    print(f"PATIENT: {patient_id}")
    print("=" * 65)

    # Verify patient belongs to this source
    assert patient_source_map[patient_id] == source_name

    # Retrieve CT archive entry
    entry = get_ct_entry(patient_id)

    assert entry["source_partition"] == source_name

    # Download CT or reuse verified local copy
    ct_path = download_ribfrac_image(patient_id)

    ct_path = Path(ct_path)

    # --------------------------------------------------------
    # 4. Verify local CT file
    # --------------------------------------------------------

    assert ct_path.is_file(), (
        f"CT file missing: {ct_path}"
    )

    assert ct_path.stat().st_size > 0, (
        f"Empty CT file: {ct_path}"
    )

    expected_size = int(entry["uncompressed_size"])

    assert ct_path.stat().st_size == expected_size, (
        f"CT size mismatch for {patient_id}"
    )

    # --------------------------------------------------------
    # 5. Load NIfTI header
    # --------------------------------------------------------

    # nib.load reads the image header without creating
    # a full CT volume array in memory.

    ct_img = nib.load(str(ct_path))

    ct_shape = ct_img.shape

    voxel_spacing = ct_img.header.get_zooms()[:3]

    data_type = str(ct_img.get_data_dtype())

    # --------------------------------------------------------
    # 6. Validate CT geometry and metadata
    # --------------------------------------------------------

    assert len(ct_shape) == 3, (
        f"{patient_id}: Expected 3D CT image."
    )

    assert all(dim > 0 for dim in ct_shape), (
        f"{patient_id}: Invalid CT dimensions."
    )

    assert all(
        np.isfinite(spacing) and spacing > 0
        for spacing in voxel_spacing
    ), f"{patient_id}: Invalid voxel spacing."

    assert np.isfinite(ct_img.affine).all(), (
        f"{patient_id}: Invalid affine matrix."
    )

    # --------------------------------------------------------
    # 7. Display examination details
    # --------------------------------------------------------

    print("\nCT successfully retrieved.")

    print("Patient ID:", patient_id)
    print("Source partition:", source_name)
    print("File path:", ct_path)
    print("CT shape:", ct_shape)
    print("Voxel spacing:", voxel_spacing)
    print("Data type:", data_type)

    print(
        "File size:",
        f"{ct_path.stat().st_size / 1024**2:.2f} MB"
    )

    print("CT retrieval: PASS")
    print("NIfTI header validation: PASS")

    # --------------------------------------------------------
    # 8. Store test results
    # --------------------------------------------------------

    ct_test_results.append({
        "patient_id": patient_id,
        "source_partition": source_name,
        "ct_path": str(ct_path),
        "shape_x": ct_shape[0],
        "shape_y": ct_shape[1],
        "shape_z": ct_shape[2],
        "spacing_x": voxel_spacing[0],
        "spacing_y": voxel_spacing[1],
        "spacing_z": voxel_spacing[2],
        "data_type": data_type,
        "file_size_mb": round(
            ct_path.stat().st_size / 1024**2, 2
        ),
        "status": "PASS"
    })

# ------------------------------------------------------------
# 9. Create CT retrieval test report
# ------------------------------------------------------------

ct_test_df = pd.DataFrame(ct_test_results)

print("\n" + "=" * 65)
print("RIBFRAC CT RETRIEVAL TEST SUMMARY")
print("=" * 65)

display(ct_test_df)

# ------------------------------------------------------------
# 10. Validate all three test results
# ------------------------------------------------------------

assert len(ct_test_df) == 3

assert ct_test_df["source_partition"].nunique() == 3

assert (ct_test_df["status"] == "PASS").all()

# ------------------------------------------------------------
# 11. Save test report
# ------------------------------------------------------------

CT_TEST_REPORT_PATH = (
    METADATA_DIR / "ribfrac_ct_retrieval_test.csv"
)

ct_test_df.to_csv(
    CT_TEST_REPORT_PATH,
    index=False
)

print("\nTest report saved:")
print(CT_TEST_REPORT_PATH)

# ------------------------------------------------------------
# 12. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RIBFRAC AUTOMATIC CT RETRIEVAL COMPLETED")
print("=" * 65)

print("Sources tested:", len(ct_test_df))
print("Successful retrievals:", len(ct_test_df))

print("\nCELL #18 VALIDATION: PASS")


RIBFRAC CT RETRIEVAL TEST CONFIGURATION
train_part1: RibFrac128
train_part2: RibFrac301
validation: RibFrac421

TESTING SOURCE: train_part1
PATIENT: RibFrac128
Verified existing CT: RibFrac128
Path: /kaggle/working/ribfrac_images/RibFrac128-image.nii.gz

CT successfully retrieved.
Patient ID: RibFrac128
Source partition: train_part1
File path: /kaggle/working/ribfrac_images/RibFrac128-image.nii.gz
CT shape: (512, 512, 337)
Voxel spacing: (np.float32(0.771484), np.float32(0.771484), np.float32(1.25))
Data type: int32
File size: 106.84 MB
CT retrieval: PASS
NIfTI header validation: PASS

TESTING SOURCE: train_part2
PATIENT: RibFrac301
Verified existing CT: RibFrac301
Path: /kaggle/working/ribfrac_images/RibFrac301-image.nii.gz

CT successfully retrieved.
Patient ID: RibFrac301
Source partition: train_part2
File path: /kaggle/working/ribfrac_images/RibFrac301-image.nii.gz
CT shape: (512, 512, 407)
Voxel spacing: (np.float32(0.7949219), np.float32(0.7949219), np.float32(1.0))
Data type: in

,patient_id,source_partition,ct_path,shape_x,shape_y,shape_z,spacing_x,spacing_y,spacing_z,data_type,file_size_mb,status
0,RibFrac128,train_part1,/kaggle/working/ribfrac_images/RibFrac128-imag...,512,512,337,0.771484,0.771484,1.25,int32,106.84,PASS
1,RibFrac301,train_part2,/kaggle/working/ribfrac_images/RibFrac301-imag...,512,512,407,0.794922,0.794922,1.00,int16,125.84,PASS
2,RibFrac421,validation,/kaggle/working/ribfrac_images/RibFrac421-imag...,512,512,325,0.878906,0.878906,1.25,int32,103.43,PASS



Test report saved:
/kaggle/working/ribfrac_metadata/ribfrac_ct_retrieval_test.csv

RIBFRAC AUTOMATIC CT RETRIEVAL COMPLETED
Sources tested: 3
Successful retrievals: 3

CELL #18 VALIDATION: PASS


## 19. Load a patient

In [30]:
# ============================================================
# CELL #19 — LOAD AND TEST A RIBFRAC PATIENT
# ============================================================

# ============================================================
# PART A — DEFINE PATIENT LOADING FUNCTION
# ============================================================

def load_patient(patient_id):
    """
    Load a RibFrac patient's CT image and fracture label.

    Supports all three RibFrac source partitions.

    Returns a dictionary containing:
        patient_id
        source_partition
        ct_path
        label_path
        ct_img
        label_img
        ct_volume
        label_volume
    """

    # --------------------------------------------------------
    # 1. Verify patient
    # --------------------------------------------------------

    patient_id = str(patient_id)

    if patient_id not in patient_source_map:
        raise ValueError(
            f"Patient not found: {patient_id}"
        )

    source_name = patient_source_map[patient_id]

    print("=" * 60)
    print("LOADING RIBFRAC PATIENT")
    print("=" * 60)

    print("Patient:", patient_id)
    print("Source:", source_name)

    # --------------------------------------------------------
    # 2. Retrieve CT and annotation
    # --------------------------------------------------------

    ct_path = Path(
        download_ribfrac_image(patient_id)
    )

    label_path = Path(
        get_label_path(patient_id)
    )

    if not ct_path.is_file():
        raise FileNotFoundError(ct_path)

    if not label_path.is_file():
        raise FileNotFoundError(label_path)

    # --------------------------------------------------------
    # 3. Load NIfTI images
    # --------------------------------------------------------

    ct_img = nib.load(str(ct_path))
    label_img = nib.load(str(label_path))

    # --------------------------------------------------------
    # 4. Verify CT and annotation geometry
    # --------------------------------------------------------

    if ct_img.ndim != 3 or label_img.ndim != 3:
        raise ValueError(
            "CT and annotation must be 3D."
        )

    if ct_img.shape != label_img.shape:
        raise ValueError(
            f"Shape mismatch: "
            f"CT={ct_img.shape}, "
            f"Label={label_img.shape}"
        )

    ct_spacing = ct_img.header.get_zooms()[:3]
    label_spacing = label_img.header.get_zooms()[:3]

    if not np.allclose(
        ct_spacing,
        label_spacing,
        rtol=1e-5,
        atol=1e-4
    ):
        raise ValueError(
            "CT and annotation voxel spacing mismatch."
        )

    if not np.allclose(
        ct_img.affine,
        label_img.affine,
        rtol=1e-5,
        atol=1e-3
    ):
        raise ValueError(
            "CT and annotation affine mismatch."
        )

    # --------------------------------------------------------
    # 5. Load voxel arrays
    # --------------------------------------------------------

    ct_volume = ct_img.get_fdata(
        dtype=np.float32,
        caching="unchanged"
    )

    label_volume = label_img.get_fdata(
        dtype=np.float32,
        caching="unchanged"
    )

    # --------------------------------------------------------
    # 6. Validate voxel values
    # --------------------------------------------------------

    if not np.isfinite(ct_volume).all():
        raise ValueError(
            "CT contains invalid voxel values."
        )

    if not np.isfinite(label_volume).all():
        raise ValueError(
            "Annotation contains invalid voxel values."
        )

    # --------------------------------------------------------
    # 7. Return patient data
    # IMPORTANT: This return belongs INSIDE the function.
    # --------------------------------------------------------

    return {
        "patient_id": patient_id,
        "source_partition": source_name,
        "ct_path": ct_path,
        "label_path": label_path,
        "ct_img": ct_img,
        "label_img": label_img,
        "ct_volume": ct_volume,
        "label_volume": label_volume
    }


# ============================================================
# PART B — TEST THE PATIENT LOADING FUNCTION
# ============================================================

patient_id = "RibFrac128"

patient = load_patient(patient_id)

print("\n" + "=" * 60)
print("PATIENT LOADING TEST")
print("=" * 60)

print("Patient:", patient["patient_id"])
print("Source:", patient["source_partition"])

print("\nCT volume:")
print("Shape:", patient["ct_volume"].shape)
print("Data type:", patient["ct_volume"].dtype)

print("\nFracture annotation:")
print("Shape:", patient["label_volume"].shape)
print("Data type:", patient["label_volume"].dtype)

print(
    "Nonzero annotation voxels:",
    np.count_nonzero(patient["label_volume"])
)

print(
    "CT voxel spacing:",
    patient["ct_img"].header.get_zooms()[:3]
)

# ============================================================
# PART C — FINAL VALIDATION
# ============================================================

assert (
    patient["ct_volume"].shape
    == patient["label_volume"].shape
)

assert patient["ct_volume"].ndim == 3
assert patient["label_volume"].ndim == 3

assert (
    patient["source_partition"]
    == patient_source_map[patient_id]
)

print("\nCT–annotation alignment: PASS")
print("Patient source matching: PASS")

print("\nCELL #19 VALIDATION: PASS")


LOADING RIBFRAC PATIENT
Patient: RibFrac128
Source: train_part1
Verified existing CT: RibFrac128
Path: /kaggle/working/ribfrac_images/RibFrac128-image.nii.gz

PATIENT LOADING TEST
Patient: RibFrac128
Source: train_part1

CT volume:
Shape: (512, 512, 337)
Data type: float32

Fracture annotation:
Shape: (512, 512, 337)
Data type: float32
Nonzero annotation voxels: 15140
CT voxel spacing: (np.float32(0.771484), np.float32(0.771484), np.float32(1.25))

CT–annotation alignment: PASS
Patient source matching: PASS

CELL #19 VALIDATION: PASS


In [25]:
patient = load_patient("RibFrac128")

Loading patient: RibFrac128
CT already exists:
/kaggle/working/ribfrac_images/RibFrac128-image.nii.gz


## 20. Patient validation

In [31]:

# ============================================================
# CELL #20 — RIBFRAC PATIENT VALIDATION
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# PART A — DEFINE PATIENT VALIDATION FUNCTION
# ============================================================

def validate_patient(patient):
    """
    Validate a loaded RibFrac patient.

    Checks:
        1. CT and label dimensions
        2. CT and label shape consistency
        3. Voxel spacing consistency
        4. Spatial affine consistency
        5. Finite voxel values
        6. Presence of fracture annotations
        7. Patient source consistency
        8. Fracture-instance IDs against metadata

    Returns:
        Dictionary of validation results.
    """

    # --------------------------------------------------------
    # 1. Retrieve patient data
    # --------------------------------------------------------

    patient_id = patient["patient_id"]
    source_name = patient["source_partition"]

    ct_img = patient["ct_img"]
    label_img = patient["label_img"]

    ct = patient["ct_volume"]
    label = patient["label_volume"]

    results = {}

    # --------------------------------------------------------
    # 2. Validate dimensions and shapes
    # --------------------------------------------------------

    results["ct_is_3d"] = bool(
        ct.ndim == 3
    )

    results["label_is_3d"] = bool(
        label.ndim == 3
    )

    results["shape_match"] = bool(
        ct.shape == label.shape
        and ct_img.shape == label_img.shape
    )

    # --------------------------------------------------------
    # 3. Validate voxel spacing
    # --------------------------------------------------------

    ct_spacing = ct_img.header.get_zooms()[:3]
    label_spacing = label_img.header.get_zooms()[:3]

    results["spacing_match"] = bool(
        np.allclose(
            ct_spacing,
            label_spacing,
            rtol=1e-5,
            atol=1e-4
        )
    )

    # --------------------------------------------------------
    # 4. Validate spatial affine matrices
    # --------------------------------------------------------

    results["affine_match"] = bool(
        np.isfinite(ct_img.affine).all()
        and np.isfinite(label_img.affine).all()
        and np.allclose(
            ct_img.affine,
            label_img.affine,
            rtol=1e-5,
            atol=1e-3
        )
    )

    # --------------------------------------------------------
    # 5. Validate voxel values
    # --------------------------------------------------------

    results["ct_finite"] = bool(
        np.isfinite(ct).all()
    )

    results["label_finite"] = bool(
        np.isfinite(label).all()
    )

    # --------------------------------------------------------
    # 6. Validate annotation presence and integer IDs
    # --------------------------------------------------------

    label_values = np.unique(label)

    results["has_annotation"] = bool(
        np.any(label_values > 0)
    )

    results["label_ids_integer"] = bool(
        np.isfinite(label_values).all()
        and np.all(label_values >= 0)
        and np.array_equal(
            label_values,
            np.rint(label_values)
        )
    )

    nifti_ids = {
        int(value)
        for value in label_values
        if value > 0 and np.isfinite(value)
    }

    # --------------------------------------------------------
    # 7. Verify patient's original source
    # --------------------------------------------------------

    results["source_match"] = bool(
        patient_id in patient_source_map
        and patient_source_map[patient_id]
        == source_name
    )

    # --------------------------------------------------------
    # 8. Retrieve metadata for this patient
    # --------------------------------------------------------

    patient_metadata = info_df.loc[
        info_df["public_id"] == patient_id
    ].copy()

    results["metadata_available"] = bool(
        not patient_metadata.empty
    )

    results["metadata_source_match"] = bool(
        not patient_metadata.empty
        and (
            patient_metadata["source_partition"]
            == source_name
        ).all()
    )

    # --------------------------------------------------------
    # 9. Compare fracture IDs against metadata
    # --------------------------------------------------------

    metadata_ids = set(
        patient_metadata.loc[
            patient_metadata["label_id"] > 0,
            "label_id"
        ].astype(int)
    )

    missing_metadata_ids = (
        nifti_ids - metadata_ids
    )

    missing_nifti_ids = (
        metadata_ids - nifti_ids
    )

    results["nifti_ids_in_metadata"] = bool(
        len(missing_metadata_ids) == 0
    )

    results["metadata_ids_in_nifti"] = bool(
        len(missing_nifti_ids) == 0
    )

    # --------------------------------------------------------
    # 10. Report annotation consistency
    # --------------------------------------------------------

    print("\nFracture annotation consistency:")
    print("NIfTI fracture IDs:", len(nifti_ids))
    print("Metadata fracture IDs:", len(metadata_ids))

    print(
        "NIfTI IDs missing from metadata:",
        sorted(missing_metadata_ids)
    )

    print(
        "Metadata IDs missing from NIfTI:",
        sorted(missing_nifti_ids)
    )

    # --------------------------------------------------------
    # 11. Return validation results
    # --------------------------------------------------------

    return results


# ============================================================
# PART B — VALIDATE THE LOADED PATIENT
# ============================================================

# Reuse the patient loaded in Cell #19.
# No second CT download is necessary.

assert "patient" in globals(), (
    "Run Cell #19 to load a patient first."
)

validation = validate_patient(patient)

print("\n" + "=" * 60)
print("RIBFRAC PATIENT VALIDATION RESULTS")
print("=" * 60)

print("Patient:", patient["patient_id"])
print("Source:", patient["source_partition"])

print("\nValidation checks:")

for check, passed in validation.items():

    status = "PASS" if passed else "FAIL"

    print(f"{check:30s}: {status}")


# ============================================================
# PART C — FINAL VALIDATION
# ============================================================

failed_checks = [
    check
    for check, passed in validation.items()
    if not passed
]

print("\n" + "=" * 60)

if not failed_checks:

    print("CELL #20 VALIDATION: PASS")
    print("Patient is ready for preprocessing.")

else:

    print("CELL #20 VALIDATION: REVIEW REQUIRED")

    print("\nFailed checks:")

    for check in failed_checks:
        print("-", check)

    print(
        "\nInvestigate these checks before "
        "using this patient for preprocessing."
    )

print("=" * 60)



Fracture annotation consistency:
NIfTI fracture IDs: 8
Metadata fracture IDs: 8
NIfTI IDs missing from metadata: []
Metadata IDs missing from NIfTI: []

RIBFRAC PATIENT VALIDATION RESULTS
Patient: RibFrac128
Source: train_part1

Validation checks:
ct_is_3d                      : PASS
label_is_3d                   : PASS
shape_match                   : PASS
spacing_match                 : PASS
affine_match                  : PASS
ct_finite                     : PASS
label_finite                  : PASS
has_annotation                : PASS
label_ids_integer             : PASS
source_match                  : PASS
metadata_available            : PASS
metadata_source_match         : PASS
nifti_ids_in_metadata         : PASS
metadata_ids_in_nifti         : PASS

CELL #20 VALIDATION: PASS
Patient is ready for preprocessing.


## 21. Display patient information

In [32]:

# ============================================================
# CELL #21 — DISPLAY RIBFRAC PATIENT INFORMATION
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# PART A — DEFINE PATIENT SUMMARY FUNCTION
# ============================================================

def patient_summary(patient):
    """
    Display a summary of a loaded RibFrac examination.

    Includes:
        - Patient ID and source partition
        - CT dimensions and voxel spacing
        - CT Hounsfield Unit statistics
        - Fracture annotation statistics
        - Fracture-instance IDs
        - Fracture subtype distribution

    Returns:
        Dictionary containing patient summary statistics.
    """

    # --------------------------------------------------------
    # 1. Retrieve patient information
    # --------------------------------------------------------

    patient_id = patient["patient_id"]
    source_name = patient["source_partition"]

    ct_img = patient["ct_img"]
    label_img = patient["label_img"]

    ct = patient["ct_volume"]
    label = patient["label_volume"]

    # --------------------------------------------------------
    # 2. Display patient identification
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print("RIBFRAC PATIENT SUMMARY")
    print("=" * 65)

    print("Patient ID:", patient_id)
    print("Source partition:", source_name)

    print("\nCT file:", patient["ct_path"])
    print("Label file:", patient["label_path"])

    # --------------------------------------------------------
    # 3. Display CT image information
    # --------------------------------------------------------

    ct_shape = ct_img.shape

    ct_spacing = ct_img.header.get_zooms()[:3]

    print("\n" + "-" * 65)
    print("CT IMAGE INFORMATION")
    print("-" * 65)

    print("CT shape:", ct_shape)
    print("CT data type:", ct.dtype)
    print("Voxel spacing (mm):", ct_spacing)

    print("\nCT intensity statistics:")

    ct_min = float(np.min(ct))
    ct_max = float(np.max(ct))
    ct_mean = float(np.mean(ct))
    ct_std = float(np.std(ct))

    print(f"Minimum intensity: {ct_min:.2f}")
    print(f"Maximum intensity: {ct_max:.2f}")
    print(f"Mean intensity: {ct_mean:.2f}")
    print(f"Standard deviation: {ct_std:.2f}")

    # --------------------------------------------------------
    # 4. Display fracture annotation information
    # --------------------------------------------------------

    print("\n" + "-" * 65)
    print("FRACTURE ANNOTATION INFORMATION")
    print("-" * 65)

    unique_labels = np.unique(label)

    fracture_ids = sorted(
        int(value)
        for value in unique_labels
        if value > 0
    )

    total_fractures = len(fracture_ids)

    nonzero_voxels = int(
        np.count_nonzero(label > 0)
    )

    print("Annotation shape:", label_img.shape)
    print("Annotation data type:", label.dtype)

    print("Unique annotation IDs:", unique_labels)

    print("Number of fracture instances:", total_fractures)
    print("Total fracture voxels:", nonzero_voxels)

    # --------------------------------------------------------
    # 5. Retrieve fracture metadata
    # --------------------------------------------------------

    patient_metadata = info_df.loc[
        info_df["public_id"] == patient_id
    ].copy()

    if patient_metadata.empty:
        raise ValueError(
            f"No fracture metadata found for {patient_id}"
        )

    # Background is not a fracture instance
    fracture_metadata = patient_metadata.loc[
        patient_metadata["label_id"] > 0
    ].copy()

    # --------------------------------------------------------
    # 6. Compare annotation IDs with metadata
    # --------------------------------------------------------

    metadata_ids = set(
        fracture_metadata["label_id"].astype(int)
    )

    nifti_ids = set(fracture_ids)

    missing_metadata = (
        nifti_ids - metadata_ids
    )

    missing_nifti = (
        metadata_ids - nifti_ids
    )

    print("\nFracture-instance consistency:")

    print("Instances in NIfTI:", len(nifti_ids))
    print("Instances in metadata:", len(metadata_ids))

    print(
        "NIfTI IDs missing from metadata:",
        sorted(missing_metadata)
    )

    print(
        "Metadata IDs missing from NIfTI:",
        sorted(missing_nifti)
    )

    # --------------------------------------------------------
    # 7. Map fracture subtype codes to category names
    # --------------------------------------------------------

    fracture_classes = {
        -1: "Unknown subtype",
         0: "Background",
         1: "Displaced fracture",
         2: "Nondisplaced fracture",
         3: "Buckle fracture",
         4: "Segmental fracture"
    }

    fracture_metadata["fracture_type"] = (
        fracture_metadata["label_code"]
        .map(fracture_classes)
        .fillna("Unrecognized code")
    )

    # --------------------------------------------------------
    # 8. Count fracture subtypes
    # --------------------------------------------------------

    print("\n" + "-" * 65)
    print("FRACTURE SUBTYPE DISTRIBUTION")
    print("-" * 65)

    subtype_counts = (
        fracture_metadata["fracture_type"]
        .value_counts()
    )

    print(subtype_counts)

    unknown_fractures = int(
        (fracture_metadata["label_code"] == -1)
        .sum()
    )

    known_fractures = int(
        fracture_metadata["label_code"]
        .isin([1, 2, 3, 4])
        .sum()
    )

    print("\nKnown subtype fractures:", known_fractures)
    print("Unknown subtype fractures:", unknown_fractures)

    # --------------------------------------------------------
    # 9. Display instance-level metadata
    # --------------------------------------------------------

    print("\nFracture annotation metadata:")

    display(
        fracture_metadata[
            [
                "label_id",
                "label_code",
                "fracture_type"
            ]
        ].sort_values("label_id")
    )

    # --------------------------------------------------------
    # 10. Create patient summary dictionary
    # --------------------------------------------------------

    summary = {
        "patient_id": patient_id,
        "source_partition": source_name,
        "ct_shape": ct_shape,
        "voxel_spacing": ct_spacing,
        "ct_min": ct_min,
        "ct_max": ct_max,
        "ct_mean": ct_mean,
        "ct_std": ct_std,
        "number_of_fracture_instances": total_fractures,
        "fracture_voxels": nonzero_voxels,
        "known_subtype_fractures": known_fractures,
        "unknown_subtype_fractures": unknown_fractures,
        "annotation_ids_match": (
            not missing_metadata
            and not missing_nifti
        )
    }

    print("\n" + "=" * 65)
    print("PATIENT SUMMARY COMPLETED")
    print("=" * 65)

    return summary


# ============================================================
# PART B — DISPLAY INFORMATION FOR THE LOADED PATIENT
# ============================================================

# Reuse the patient loaded in Cell #19.
# No additional CT download is required.

assert "patient" in globals(), (
    "Run Cell #19 before running Cell #21."
)

summary = patient_summary(patient)


# ============================================================
# PART C — FINAL SUMMARY CHECKS
# ============================================================

print("\n" + "=" * 65)
print("CELL #21 — FINAL VERIFICATION")
print("=" * 65)

assert summary["patient_id"] == patient["patient_id"]

assert (
    summary["source_partition"]
    == patient_source_map[patient["patient_id"]]
)

if summary["annotation_ids_match"]:

    print("Fracture-instance IDs: PASS")

else:

    print("Fracture-instance IDs: REVIEW REQUIRED")

    print(
        "Check differences between NIfTI annotation "
        "IDs and the metadata."
    )

print("\nCELL #21 EXECUTION COMPLETED")



RIBFRAC PATIENT SUMMARY
Patient ID: RibFrac128
Source partition: train_part1

CT file: /kaggle/working/ribfrac_images/RibFrac128-image.nii.gz
Label file: /kaggle/working/ribfrac_labels/Part1/RibFrac128-label.nii.gz

-----------------------------------------------------------------
CT IMAGE INFORMATION
-----------------------------------------------------------------
CT shape: (512, 512, 337)
CT data type: float32
Voxel spacing (mm): (np.float32(0.771484), np.float32(0.771484), np.float32(1.25))

CT intensity statistics:
Minimum intensity: -3024.00
Maximum intensity: 1686.00
Mean intensity: -1077.17
Standard deviation: 1110.61

-----------------------------------------------------------------
FRACTURE ANNOTATION INFORMATION
-----------------------------------------------------------------
Annotation shape: (512, 512, 337)
Annotation data type: float32
Unique annotation IDs: [0. 1. 2. 3. 4. 5. 6. 7. 8.]
Number of fracture instances: 8
Total fracture voxels: 15140

Fracture-instance cons

,label_id,label_code,fracture_type
1334,1,-1,Unknown subtype
1335,2,-1,Unknown subtype
1336,3,-1,Unknown subtype
1337,4,-1,Unknown subtype
1338,5,-1,Unknown subtype
1339,6,-1,Unknown subtype
1340,7,3,Buckle fracture
1341,8,-1,Unknown subtype



PATIENT SUMMARY COMPLETED

CELL #21 — FINAL VERIFICATION
Fracture-instance IDs: PASS

CELL #21 EXECUTION COMPLETED


## 22. Associate annotation IDs with fracture METADATA

In [33]:

# ============================================================
# CELL #22 — ASSOCIATE ANNOTATION IDs WITH FRACTURE METADATA
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# PART A — RETRIEVE PATIENT INFORMATION
# ============================================================

# Reuse the patient loaded in Cell #19

if "patient" not in globals():
    raise RuntimeError(
        "Please run Cell #19 to load a patient first."
    )

patient_id = patient["patient_id"]
source_name = patient["source_partition"]

print("=" * 65)
print("RIBFRAC ANNOTATION–METADATA ASSOCIATION")
print("=" * 65)

print("Patient ID:", patient_id)
print("Source partition:", source_name)


# ============================================================
# PART B — RETRIEVE PATIENT FRACTURE METADATA
# ============================================================

required_columns = [
    "public_id",
    "label_id",
    "label_code",
    "source_partition"
]

assert all(
    col in info_df.columns
    for col in required_columns
), "Required metadata columns are missing."

patient_info = info_df.loc[
    (info_df["public_id"] == patient_id) &
    (info_df["source_partition"] == source_name)
].copy()

if patient_info.empty:
    raise ValueError(
        f"No metadata found for patient {patient_id}"
    )

patient_info["label_id"] = (
    patient_info["label_id"].astype(int)
)

patient_info["label_code"] = (
    patient_info["label_code"].astype(int)
)

if patient_info["label_id"].duplicated().any():
    raise ValueError(
        f"Duplicate annotation IDs for {patient_id}"
    )

print("\nPatient metadata records:", len(patient_info))

print("\nOriginal annotation metadata:")
display(patient_info.sort_values("label_id"))


# ============================================================
# PART C — IDENTIFY ANNOTATION IDs IN THE NIFTI MASK
# ============================================================

label_volume = patient["label_volume"]

unique_values = np.unique(label_volume)

# Verify that voxel values represent integer instance IDs
if not (
    np.isfinite(unique_values).all()
    and np.all(unique_values >= 0)
    and np.array_equal(
        unique_values,
        np.rint(unique_values)
    )
):
    raise ValueError(
        "NIfTI label contains invalid instance IDs."
    )

nifti_fracture_ids = {
    int(value)
    for value in unique_values
    if value > 0
}

metadata_fracture_ids = set(
    patient_info.loc[
        patient_info["label_id"] > 0,
        "label_id"
    ]
)

print("\n" + "-" * 65)
print("FRACTURE INSTANCE IDS")
print("-" * 65)

print("NIfTI fracture IDs:")
print(sorted(nifti_fracture_ids))

print("\nMetadata fracture IDs:")
print(sorted(metadata_fracture_ids))


# ============================================================
# PART D — ASSOCIATE INSTANCE IDS WITH FRACTURE TYPES
# ============================================================

FRACTURE_CLASSES = {
    -1: "Unknown subtype",
     0: "Background",
     1: "Displaced fracture",
     2: "Nondisplaced fracture",
     3: "Buckle fracture",
     4: "Segmental fracture"
}

patient_info["fracture_type"] = (
    patient_info["label_code"]
    .map(FRACTURE_CLASSES)
    .fillna("Unrecognized subtype")
)

# Exclude background from the fracture-instance table
patient_fracture_table = patient_info.loc[
    patient_info["label_id"] > 0
].copy()

patient_fracture_table["present_in_nifti"] = (
    patient_fracture_table["label_id"]
    .isin(nifti_fracture_ids)
)

patient_fracture_table = (
    patient_fracture_table[
        [
            "public_id",
            "source_partition",
            "label_id",
            "label_code",
            "fracture_type",
            "present_in_nifti"
        ]
    ]
    .sort_values("label_id")
    .reset_index(drop=True)
)

print("\n" + "-" * 65)
print("FRACTURE INSTANCE–SUBTYPE ASSOCIATION")
print("-" * 65)

display(patient_fracture_table)


# ============================================================
# PART E — VERIFY ANNOTATION CONSISTENCY
# ============================================================

ids_missing_from_metadata = (
    nifti_fracture_ids - metadata_fracture_ids
)

ids_missing_from_nifti = (
    metadata_fracture_ids - nifti_fracture_ids
)

print("\n" + "-" * 65)
print("ANNOTATION CONSISTENCY")
print("-" * 65)

print(
    "Fracture instances in NIfTI:",
    len(nifti_fracture_ids)
)

print(
    "Fracture instances in metadata:",
    len(metadata_fracture_ids)
)

print(
    "\nIDs missing from metadata:",
    sorted(ids_missing_from_metadata)
)

print(
    "IDs missing from NIfTI:",
    sorted(ids_missing_from_nifti)
)

annotation_ids_match = (
    len(ids_missing_from_metadata) == 0
    and len(ids_missing_from_nifti) == 0
)


# ============================================================
# PART F — DISPLAY FRACTURE SUBTYPE DISTRIBUTION
# ============================================================

print("\n" + "-" * 65)
print("FRACTURE SUBTYPE DISTRIBUTION")
print("-" * 65)

subtype_distribution = (
    patient_fracture_table["fracture_type"]
    .value_counts()
)

print(subtype_distribution)

known_fractures = int(
    patient_fracture_table["label_code"]
    .isin([1, 2, 3, 4])
    .sum()
)

unknown_fractures = int(
    (
        patient_fracture_table["label_code"] == -1
    ).sum()
)

print("\nKnown subtype fractures:", known_fractures)
print("Unknown subtype fractures:", unknown_fractures)


# ============================================================
# PART G — FINAL VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("CELL #22 — FINAL VERIFICATION")
print("=" * 65)

if annotation_ids_match:
    print("Annotation–metadata association: PASS")
else:
    print("Annotation–metadata association: REVIEW REQUIRED")
    print(
        "Investigate missing instance IDs before preprocessing."
    )

print("\nCELL #22 EXECUTION COMPLETED")


RIBFRAC ANNOTATION–METADATA ASSOCIATION
Patient ID: RibFrac128
Source partition: train_part1

Patient metadata records: 9

Original annotation metadata:


,public_id,label_id,label_code,source_partition
1333,RibFrac128,0,0,train_part1
1334,RibFrac128,1,-1,train_part1
1335,RibFrac128,2,-1,train_part1
1336,RibFrac128,3,-1,train_part1
1337,RibFrac128,4,-1,train_part1
1338,RibFrac128,5,-1,train_part1
1339,RibFrac128,6,-1,train_part1
1340,RibFrac128,7,3,train_part1
1341,RibFrac128,8,-1,train_part1



-----------------------------------------------------------------
FRACTURE INSTANCE IDS
-----------------------------------------------------------------
NIfTI fracture IDs:
[1, 2, 3, 4, 5, 6, 7, 8]

Metadata fracture IDs:
[1, 2, 3, 4, 5, 6, 7, 8]

-----------------------------------------------------------------
FRACTURE INSTANCE–SUBTYPE ASSOCIATION
-----------------------------------------------------------------


,public_id,source_partition,label_id,label_code,fracture_type,present_in_nifti
0,RibFrac128,train_part1,1,-1,Unknown subtype,True
1,RibFrac128,train_part1,2,-1,Unknown subtype,True
2,RibFrac128,train_part1,3,-1,Unknown subtype,True
3,RibFrac128,train_part1,4,-1,Unknown subtype,True
4,RibFrac128,train_part1,5,-1,Unknown subtype,True
5,RibFrac128,train_part1,6,-1,Unknown subtype,True
6,RibFrac128,train_part1,7,3,Buckle fracture,True
7,RibFrac128,train_part1,8,-1,Unknown subtype,True



-----------------------------------------------------------------
ANNOTATION CONSISTENCY
-----------------------------------------------------------------
Fracture instances in NIfTI: 8
Fracture instances in metadata: 8

IDs missing from metadata: []
IDs missing from NIfTI: []

-----------------------------------------------------------------
FRACTURE SUBTYPE DISTRIBUTION
-----------------------------------------------------------------
fracture_type
Unknown subtype    7
Buckle fracture    1
Name: count, dtype: int64

Known subtype fractures: 1
Unknown subtype fractures: 7

CELL #22 — FINAL VERIFICATION
Annotation–metadata association: PASS

CELL #22 EXECUTION COMPLETED


## 23. Create a patient annotation dictionary

In [35]:

# ============================================================
# CELL #23 — CREATE PATIENT ANNOTATION DICTIONARY
# ============================================================

# ============================================================
# PART A — DEFINE ANNOTATION METADATA FUNCTION
# ============================================================

def get_patient_annotation_metadata(patient_id):
    """
    Create a dictionary linking RibFrac fracture-instance
    IDs to their corresponding fracture subtype codes.

    Supports all 500 publicly annotated RibFrac patients.

    Returns
    -------
    dict:
        {
            label_id: {
                "label_code": int,
                "fracture_type": str
            }
        }

    Background (label_id = 0) is excluded.
    Unknown subtypes (label_code = -1) are retained.
    """

    # --------------------------------------------------------
    # 1. Validate patient ID
    # --------------------------------------------------------

    patient_id = str(patient_id)

    if patient_id not in patient_source_map:
        raise ValueError(
            f"Unknown RibFrac patient: {patient_id}"
        )

    source_name = patient_source_map[patient_id]

    # --------------------------------------------------------
    # 2. Retrieve patient metadata
    # --------------------------------------------------------

    rows = info_df.loc[
        (info_df["public_id"] == patient_id) &
        (info_df["source_partition"] == source_name)
    ].copy()

    if rows.empty:
        raise ValueError(
            f"No annotation metadata for {patient_id}"
        )

    # --------------------------------------------------------
    # 3. Define fracture subtype categories
    # --------------------------------------------------------

    fracture_classes = {
        -1: "Unknown subtype",
         1: "Displaced fracture",
         2: "Nondisplaced fracture",
         3: "Buckle fracture",
         4: "Segmental fracture"
    }

    # --------------------------------------------------------
    # 4. Exclude background annotation
    # --------------------------------------------------------

    rows = rows.loc[
        rows["label_id"] > 0
    ].copy()

    # --------------------------------------------------------
    # 5. Validate fracture metadata
    # --------------------------------------------------------

    if rows["label_id"].duplicated().any():
        raise ValueError(
            f"Duplicate fracture IDs for {patient_id}"
        )

    invalid_codes = set(
        rows["label_code"].dropna().astype(int)
    ) - set(fracture_classes)

    if invalid_codes:
        raise ValueError(
            f"Unexpected fracture subtype codes: "
            f"{invalid_codes}"
        )

    if rows["label_code"].isna().any():
        raise ValueError(
            f"Missing fracture subtype codes for {patient_id}"
        )

    # --------------------------------------------------------
    # 6. Build patient annotation dictionary
    # --------------------------------------------------------

    annotation_dict = {}

    for row in rows.itertuples(index=False):

        annotation_id = int(row.label_id)
        label_code = int(row.label_code)

        annotation_dict[annotation_id] = {
            "label_code": label_code,
            "fracture_type": fracture_classes[label_code]
        }

    return annotation_dict


# ============================================================
# PART B — CREATE DICTIONARY FOR LOADED PATIENT
# ============================================================

# Reuse the patient loaded in Cell #19.

if "patient" not in globals():
    raise RuntimeError(
        "Please run Cell #19 first."
    )

patient_id = patient["patient_id"]

annotation_metadata = (
    get_patient_annotation_metadata(patient_id)
)

print("=" * 65)
print("RIBFRAC PATIENT ANNOTATION DICTIONARY")
print("=" * 65)

print("Patient:", patient_id)
print("Source:", patient_source_map[patient_id])

print(
    "\nNumber of fracture instances:",
    len(annotation_metadata)
)

print("\nAnnotation dictionary:")

for annotation_id, details in sorted(
    annotation_metadata.items()
):

    print(
        f"Fracture ID: {annotation_id} | "
        f"Code: {details['label_code']} | "
        f"Type: {details['fracture_type']}"
    )


# ============================================================
# PART C — VALIDATE DICTIONARY AGAINST NIFTI LABELS
# ============================================================

# Extract positive fracture-instance IDs from the NIfTI mask.

label_volume = patient["label_volume"]

nifti_ids = set(
    np.unique(
        label_volume[label_volume > 0]
    ).astype(int)
)

metadata_ids = set(
    annotation_metadata.keys()
)

# ------------------------------------------------------------
# 7. Compare fracture-instance identifiers
# ------------------------------------------------------------

missing_metadata = nifti_ids - metadata_ids
missing_nifti = metadata_ids - nifti_ids

print("\n" + "-" * 65)
print("ANNOTATION DICTIONARY VALIDATION")
print("-" * 65)

print("NIfTI fracture IDs:", len(nifti_ids))
print("Metadata fracture IDs:", len(metadata_ids))

print(
    "NIfTI IDs missing from metadata:",
    sorted(missing_metadata)
)

print(
    "Metadata IDs missing from NIfTI:",
    sorted(missing_nifti)
)

# ------------------------------------------------------------
# 8. Count known and unknown fracture subtypes
# ------------------------------------------------------------

known_fractures = sum(
    details["label_code"] in [1, 2, 3, 4]
    for details in annotation_metadata.values()
)

unknown_fractures = sum(
    details["label_code"] == -1
    for details in annotation_metadata.values()
)

print("\nKnown subtype fractures:", known_fractures)
print("Unknown subtype fractures:", unknown_fractures)

assert (
    known_fractures + unknown_fractures
    == len(annotation_metadata)
)

# ------------------------------------------------------------
# 9. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)

if not missing_metadata and not missing_nifti:

    print("Annotation dictionary consistency: PASS")
    print("CELL #23 VALIDATION: PASS")

else:

    print("Annotation dictionary consistency: REVIEW REQUIRED")
    print("CELL #23 VALIDATION: REVIEW REQUIRED")

print("=" * 65)


RIBFRAC PATIENT ANNOTATION DICTIONARY
Patient: RibFrac128
Source: train_part1

Number of fracture instances: 8

Annotation dictionary:
Fracture ID: 1 | Code: -1 | Type: Unknown subtype
Fracture ID: 2 | Code: -1 | Type: Unknown subtype
Fracture ID: 3 | Code: -1 | Type: Unknown subtype
Fracture ID: 4 | Code: -1 | Type: Unknown subtype
Fracture ID: 5 | Code: -1 | Type: Unknown subtype
Fracture ID: 6 | Code: -1 | Type: Unknown subtype
Fracture ID: 7 | Code: 3 | Type: Buckle fracture
Fracture ID: 8 | Code: -1 | Type: Unknown subtype

-----------------------------------------------------------------
ANNOTATION DICTIONARY VALIDATION
-----------------------------------------------------------------
NIfTI fracture IDs: 8
Metadata fracture IDs: 8
NIfTI IDs missing from metadata: []
Metadata IDs missing from NIfTI: []

Known subtype fractures: 1
Unknown subtype fractures: 7

Annotation dictionary consistency: PASS
CELL #23 VALIDATION: PASS


## 24. Check annotation IDs against metadata

In [37]:

# ============================================================
# CELL #24 — CHECK ANNOTATION IDs AGAINST METADATA
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# PART A — RETRIEVE THE LOADED PATIENT
# ============================================================

# Reuse the patient loaded in Cell #19.

if "patient" not in globals():
    raise RuntimeError(
        "Run Cell #19 to load a patient first."
    )

patient_id = patient["patient_id"]
source_name = patient["source_partition"]

label_volume = patient["label_volume"]

print("=" * 65)
print("RIBFRAC ANNOTATION ID VALIDATION")
print("=" * 65)

print("Patient:", patient_id)
print("Source partition:", source_name)


# ============================================================
# PART B — EXTRACT FRACTURE IDs FROM NIFTI
# ============================================================

unique_values = np.unique(label_volume)

# Check that annotation values are valid instance IDs.

if not np.isfinite(unique_values).all():
    raise ValueError(
        "NIfTI annotation contains non-finite values."
    )

if np.any(unique_values < 0):
    raise ValueError(
        "NIfTI annotation contains negative instance IDs."
    )

if not np.array_equal(
    unique_values,
    np.rint(unique_values)
):
    raise ValueError(
        "NIfTI annotation contains non-integer instance IDs."
    )

# Exclude background label 0.

label_ids_in_nifti = {
    int(value)
    for value in unique_values
    if value > 0
}

print("\nFracture IDs in NIfTI:")
print(sorted(label_ids_in_nifti))

print(
    "Total NIfTI fracture instances:",
    len(label_ids_in_nifti)
)


# ============================================================
# PART C — RETRIEVE FRACTURE IDs FROM METADATA
# ============================================================

# Refresh annotation metadata for the current patient.
# Uses the function created in Cell #23.

annotation_metadata = (
    get_patient_annotation_metadata(patient_id)
)

metadata_ids = set(
    annotation_metadata.keys()
)

print("\nFracture IDs in metadata:")
print(sorted(metadata_ids))

print(
    "Total metadata fracture instances:",
    len(metadata_ids)
)


# ============================================================
# PART D — COMPARE INSTANCE IDs
# ============================================================

# Fracture IDs found in NIfTI but missing in metadata
missing_from_metadata = (
    label_ids_in_nifti - metadata_ids
)

# Fracture IDs found in metadata but absent in NIfTI
missing_from_nifti = (
    metadata_ids - label_ids_in_nifti
)

# Fracture IDs found in both
matching_ids = (
    label_ids_in_nifti & metadata_ids
)

print("\n" + "-" * 65)
print("ANNOTATION ID COMPARISON")
print("-" * 65)

print("Matching fracture IDs:", len(matching_ids))

print("\nNIfTI IDs missing from metadata:")
print(sorted(missing_from_metadata))

print("\nMetadata IDs missing from NIfTI:")
print(sorted(missing_from_nifti))


# ============================================================
# PART E — BUILD ANNOTATION COMPARISON TABLE
# ============================================================

all_fracture_ids = sorted(
    label_ids_in_nifti | metadata_ids
)

comparison_records = []

for fracture_id in all_fracture_ids:

    in_nifti = fracture_id in label_ids_in_nifti
    in_metadata = fracture_id in metadata_ids

    if in_nifti and in_metadata:
        status = "MATCH"

    elif in_nifti:
        status = "NIFTI ONLY"

    else:
        status = "METADATA ONLY"

    details = annotation_metadata.get(
        fracture_id, {}
    )

    comparison_records.append({
        "patient_id": patient_id,
        "source_partition": source_name,
        "label_id": fracture_id,
        "present_in_nifti": in_nifti,
        "present_in_metadata": in_metadata,
        "label_code": details.get("label_code"),
        "fracture_type": details.get("fracture_type"),
        "status": status
    })

annotation_comparison_df = pd.DataFrame(
    comparison_records,
    columns=[
        "patient_id",
        "source_partition",
        "label_id",
        "present_in_nifti",
        "present_in_metadata",
        "label_code",
        "fracture_type",
        "status"
    ]
)

print("\n" + "-" * 65)
print("FRACTURE INSTANCE COMPARISON TABLE")
print("-" * 65)

display(annotation_comparison_df)


# ============================================================
# PART F — CHECK UNKNOWN FRACTURE SUBTYPES
# ============================================================

# Unknown subtype (-1) is still a fracture.
# It must not be removed from binary segmentation.

unknown_ids = {
    fracture_id
    for fracture_id, details
    in annotation_metadata.items()
    if details["label_code"] == -1
}

unknown_present = (
    unknown_ids & label_ids_in_nifti
)

print("\n" + "-" * 65)
print("UNKNOWN FRACTURE SUBTYPE CHECK")
print("-" * 65)

print("Unknown-subtype IDs in metadata:", len(unknown_ids))

print(
    "Unknown-subtype IDs present in NIfTI:",
    len(unknown_present)
)

print(
    "Unknown-subtype IDs absent from NIfTI:",
    sorted(unknown_ids - label_ids_in_nifti)
)


# ============================================================
# PART G — SAVE THE VALIDATION REPORT
# ============================================================

ANNOTATION_CHECK_PATH = (
    METADATA_DIR /
    f"{patient_id}_annotation_id_validation.csv"
)

annotation_comparison_df.to_csv(
    ANNOTATION_CHECK_PATH,
    index=False
)

print("\nValidation report saved:")
print(ANNOTATION_CHECK_PATH)


# ============================================================
# PART H — FINAL VALIDATION
# ============================================================

annotation_ids_match = (
    len(missing_from_metadata) == 0
    and len(missing_from_nifti) == 0
)

print("\n" + "=" * 65)
print("CELL #24 — FINAL VALIDATION")
print("=" * 65)

print("Patient:", patient_id)
print("NIfTI fracture instances:", len(label_ids_in_nifti))
print("Metadata fracture instances:", len(metadata_ids))
print("Matching instances:", len(matching_ids))

if annotation_ids_match:

    print("\nAnnotation ID consistency: PASS")
    print("CELL #24 VALIDATION: PASS")

else:

    print("\nAnnotation ID consistency: REVIEW REQUIRED")
    print("CELL #24 VALIDATION: REVIEW REQUIRED")

    print(
        "\nInvestigate mismatched fracture IDs "
        "before using this examination for training."
    )


RIBFRAC ANNOTATION ID VALIDATION
Patient: RibFrac128
Source partition: train_part1

Fracture IDs in NIfTI:
[1, 2, 3, 4, 5, 6, 7, 8]
Total NIfTI fracture instances: 8

Fracture IDs in metadata:
[1, 2, 3, 4, 5, 6, 7, 8]
Total metadata fracture instances: 8

-----------------------------------------------------------------
ANNOTATION ID COMPARISON
-----------------------------------------------------------------
Matching fracture IDs: 8

NIfTI IDs missing from metadata:
[]

Metadata IDs missing from NIfTI:
[]

-----------------------------------------------------------------
FRACTURE INSTANCE COMPARISON TABLE
-----------------------------------------------------------------


,patient_id,source_partition,label_id,present_in_nifti,present_in_metadata,label_code,fracture_type,status
0,RibFrac128,train_part1,1,True,True,-1,Unknown subtype,MATCH
1,RibFrac128,train_part1,2,True,True,-1,Unknown subtype,MATCH
2,RibFrac128,train_part1,3,True,True,-1,Unknown subtype,MATCH
3,RibFrac128,train_part1,4,True,True,-1,Unknown subtype,MATCH
4,RibFrac128,train_part1,5,True,True,-1,Unknown subtype,MATCH
5,RibFrac128,train_part1,6,True,True,-1,Unknown subtype,MATCH
6,RibFrac128,train_part1,7,True,True,3,Buckle fracture,MATCH
7,RibFrac128,train_part1,8,True,True,-1,Unknown subtype,MATCH



-----------------------------------------------------------------
UNKNOWN FRACTURE SUBTYPE CHECK
-----------------------------------------------------------------
Unknown-subtype IDs in metadata: 7
Unknown-subtype IDs present in NIfTI: 7
Unknown-subtype IDs absent from NIfTI: []

Validation report saved:
/kaggle/working/ribfrac_metadata/RibFrac128_annotation_id_validation.csv

CELL #24 — FINAL VALIDATION
Patient: RibFrac128
NIfTI fracture instances: 8
Metadata fracture instances: 8
Matching instances: 8

Annotation ID consistency: PASS
CELL #24 VALIDATION: PASS


## 25. Patient-level splitting

In [38]:

# ============================================================
# CELL #25 — RIBFRAC PATIENT-LEVEL SPLITTING
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# ============================================================
# PART A — CONFIGURATION
# ============================================================

RANDOM_SEED = 42

EXPECTED_SPLITS = {
    "train": 350,
    "validation": 75,
    "test": 75
}

# If a verified 500-patient split already exists in
# Notebook 02's saved Kaggle dataset, enter its exact path.
# Otherwise, leave as None.

EXISTING_SPLIT_PATH = None

# Canonical output for all subsequent notebooks
SPLIT_PATH = (
    METADATA_DIR / "patient_split_500.csv"
)

# ============================================================
# PART B — VERIFY PATIENT REGISTRY
# ============================================================

patient_ids = sorted(
    info_df["public_id"].astype(str).unique()
)

patients = np.array(patient_ids)

assert len(patients) == 500
assert len(set(patient_ids)) == 500
assert len(patient_source_map) == 500

print("=" * 60)
print("RIBFRAC PATIENT-LEVEL SPLITTING")
print("=" * 60)

print("Total patients:", len(patients))
print("Random seed:", RANDOM_SEED)

# ============================================================
# PART C — LOAD AN EXISTING SPLIT IF AVAILABLE
# ============================================================

if EXISTING_SPLIT_PATH is not None:

    selected_path = Path(EXISTING_SPLIT_PATH)

    if not selected_path.is_file():
        raise FileNotFoundError(selected_path)

elif SPLIT_PATH.is_file():

    selected_path = SPLIT_PATH

else:

    selected_path = None

if selected_path is not None:

    print("\nLoading existing patient split:")
    print(selected_path)

    split_df = pd.read_csv(selected_path)

    if not {"patient_id", "split"}.issubset(
        split_df.columns
    ):
        raise ValueError(
            "Existing split must contain "
            "'patient_id' and 'split' columns."
        )

    split_df = split_df[
        ["patient_id", "split"]
    ].copy()

    split_df["patient_id"] = (
        split_df["patient_id"].astype(str)
    )

    split_df["split"] = (
        split_df["split"].astype(str)
        .str.strip()
        .str.lower()
        .replace({"val": "validation"})
    )

    print("Existing assignments loaded.")

else:

    print("\nNo saved 500-patient split selected.")
    print("Generating reproducible 350/75/75 split.")

    # --------------------------------------------------------
    # 1. Select 75 patients for independent testing
    # --------------------------------------------------------

    train_val_patients, test_patients = train_test_split(
        patients,
        test_size=75,
        random_state=RANDOM_SEED,
        shuffle=True
    )

    # --------------------------------------------------------
    # 2. Select 75 patients for validation
    # --------------------------------------------------------

    train_patients, val_patients = train_test_split(
        train_val_patients,
        test_size=75,
        random_state=RANDOM_SEED,
        shuffle=True
    )

    # --------------------------------------------------------
    # 3. Create patient-level assignment table
    # --------------------------------------------------------

    split_df = pd.DataFrame({
        "patient_id": (
            list(train_patients)
            + list(val_patients)
            + list(test_patients)
        ),
        "split": (
            ["train"] * len(train_patients)
            + ["validation"] * len(val_patients)
            + ["test"] * len(test_patients)
        )
    })

# ============================================================
# PART D — VERIFY SPLIT ASSIGNMENTS
# ============================================================

assert len(split_df) == 500, (
    "Split must contain exactly 500 patient records."
)

assert split_df["patient_id"].notna().all()
assert split_df["split"].notna().all()

assert not split_df["patient_id"].duplicated().any(), (
    "Duplicate patient assignments detected."
)

assert set(split_df["patient_id"]) == set(patient_ids), (
    "Split patient IDs differ from the 500-patient registry."
)

assert set(split_df["split"]) == set(EXPECTED_SPLITS), (
    "Unexpected split names detected."
)

for split_name, expected in EXPECTED_SPLITS.items():

    actual = int(
        (split_df["split"] == split_name).sum()
    )

    assert actual == expected, (
        f"{split_name}: Expected {expected}, "
        f"found {actual}"
    )

# ============================================================
# PART E — CREATE TRAIN, VALIDATION AND TEST ARRAYS
# ============================================================

train_patients = np.array(
    split_df.loc[
        split_df["split"] == "train",
        "patient_id"
    ].tolist()
)

val_patients = np.array(
    split_df.loc[
        split_df["split"] == "validation",
        "patient_id"
    ].tolist()
)

test_patients = np.array(
    split_df.loc[
        split_df["split"] == "test",
        "patient_id"
    ].tolist()
)

# Preserve variables used by subsequent notebook cells
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

# ============================================================
# PART F — CHECK FOR PATIENT LEAKAGE
# ============================================================

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

assert len(train_set | val_set | test_set) == 500

print("\nPatient leakage checks: PASS")

# ============================================================
# PART G — ATTACH ORIGINAL DATA SOURCE
# ============================================================

split_df["source_partition"] = (
    split_df["patient_id"].map(patient_source_map)
)

assert split_df["source_partition"].notna().all()

split_df = (
    split_df
    .sort_values(["split", "patient_id"])
    .reset_index(drop=True)
)

# ============================================================
# PART H — SAVE CANONICAL SPLIT
# ============================================================

# Never silently overwrite conflicting saved assignments.
if SPLIT_PATH.is_file():

    old_df = pd.read_csv(SPLIT_PATH)

    old_map = dict(zip(
        old_df["patient_id"].astype(str),
        old_df["split"].astype(str)
            .str.lower()
            .replace({"val": "validation"})
    ))

    new_map = dict(zip(
        split_df["patient_id"],
        split_df["split"]
    ))

    if old_map != new_map:
        raise RuntimeError(
            "Existing canonical split has different "
            "patient assignments. Resolve the conflict "
            "before saving."
        )

split_df.to_csv(
    SPLIT_PATH,
    index=False
)

# ============================================================
# PART I — DISPLAY FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("FINAL RIBFRAC PATIENT SPLIT")
print("=" * 60)

print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Testing patients:", len(test_patients))

print("\nSource distribution by split:")

source_table = pd.crosstab(
    split_df["split"],
    split_df["source_partition"]
)

display(source_table)

print("\nFirst 10 patient assignments:")
display(split_df.head(10))

print("\nCanonical split saved:")
print(SPLIT_PATH)

print("\nCELL #25 VALIDATION: PASS")


RIBFRAC PATIENT-LEVEL SPLITTING
Total patients: 500
Random seed: 42

No saved 500-patient split selected.
Generating reproducible 350/75/75 split.

Patient leakage checks: PASS

FINAL RIBFRAC PATIENT SPLIT
Training patients: 350
Validation patients: 75
Testing patients: 75

Source distribution by split:


source_partition,train_part1,train_part2,validation
split,,,
test,50,11,14
train,212,87,51
validation,38,22,15



First 10 patient assignments:


,patient_id,split,source_partition
0,RibFrac1,test,train_part1
1,RibFrac100,test,train_part1
2,RibFrac107,test,train_part1
3,RibFrac109,test,train_part1
4,RibFrac115,test,train_part1
5,RibFrac119,test,train_part1
6,RibFrac126,test,train_part1
7,RibFrac129,test,train_part1
8,RibFrac134,test,train_part1
9,RibFrac140,test,train_part1



Canonical split saved:
/kaggle/working/ribfrac_metadata/patient_split_500.csv

CELL #25 VALIDATION: PASS


## 26. Verify no patient leakage

In [39]:

# ============================================================
# CELL #26 — VERIFY NO PATIENT-LEVEL DATA LEAKAGE
# ============================================================

# ============================================================
# PART A — VERIFY PATIENT SPLIT ARRAYS
# ============================================================

print("=" * 65)
print("RIBFRAC PATIENT-LEVEL DATA LEAKAGE VERIFICATION")
print("=" * 65)

train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

print("\nPatient counts:")
print("Training:", len(train_patients))
print("Validation:", len(val_patients))
print("Testing:", len(test_patients))

assert len(train_patients) == 350
assert len(val_patients) == 75
assert len(test_patients) == 75

# Verify no duplicates within individual splits
assert len(train_set) == len(train_patients)
assert len(val_set) == len(val_patients)
assert len(test_set) == len(test_patients)

print("\nPatient counts and uniqueness: PASS")


# ============================================================
# PART B — CHECK INTERSECTION BETWEEN SPLITS
# ============================================================

train_val_overlap = train_set & val_set
train_test_overlap = train_set & test_set
val_test_overlap = val_set & test_set

print("\n" + "-" * 65)
print("PATIENT OVERLAP ANALYSIS")
print("-" * 65)

print("Train / Validation overlap:", len(train_val_overlap))
print("Train / Test overlap:", len(train_test_overlap))
print("Validation / Test overlap:", len(val_test_overlap))

assert not train_val_overlap, (
    f"Training/validation leakage: {train_val_overlap}"
)

assert not train_test_overlap, (
    f"Training/test leakage: {train_test_overlap}"
)

assert not val_test_overlap, (
    f"Validation/test leakage: {val_test_overlap}"
)

print("\nPatient overlap verification: PASS")


# ============================================================
# PART C — VERIFY COMPLETE PATIENT COVERAGE
# ============================================================

all_split_patients = (
    train_set | val_set | test_set
)

metadata_patients = set(
    info_df["public_id"].astype(str).unique()
)

assert len(all_split_patients) == 500
assert len(metadata_patients) == 500

missing_patients = (
    metadata_patients - all_split_patients
)

unexpected_patients = (
    all_split_patients - metadata_patients
)

print("\n" + "-" * 65)
print("PATIENT COVERAGE")
print("-" * 65)

print("Metadata patients:", len(metadata_patients))
print("Assigned patients:", len(all_split_patients))
print("Missing patients:", len(missing_patients))
print("Unexpected patients:", len(unexpected_patients))

assert not missing_patients
assert not unexpected_patients

print("\nPatient coverage verification: PASS")


# ============================================================
# PART D — VERIFY SAVED PATIENT SPLIT FILE
# ============================================================

split_path = Path(SPLIT_PATH)

if not split_path.is_file():
    raise FileNotFoundError(
        f"Saved patient split not found: {split_path}"
    )

saved_split_df = pd.read_csv(split_path)

required_columns = {
    "patient_id",
    "split",
    "source_partition"
}

assert required_columns.issubset(
    saved_split_df.columns
), "Saved split is missing required columns."

assert len(saved_split_df) == 500

assert saved_split_df["patient_id"].notna().all()
assert saved_split_df["split"].notna().all()
assert saved_split_df["source_partition"].notna().all()

assert not saved_split_df[
    "patient_id"
].duplicated().any()

saved_split_df["patient_id"] = (
    saved_split_df["patient_id"].astype(str)
)

saved_split_df["split"] = (
    saved_split_df["split"].astype(str)
    .str.strip().str.lower()
)

# Verify saved assignments against current split arrays
expected_assignments = {
    **{pid: "train" for pid in train_set},
    **{pid: "validation" for pid in val_set},
    **{pid: "test" for pid in test_set}
}

saved_assignments = dict(zip(
    saved_split_df["patient_id"],
    saved_split_df["split"]
))

assert saved_assignments == expected_assignments, (
    "Saved patient assignments differ "
    "from the current split arrays."
)

print("\nSaved split consistency: PASS")


# ============================================================
# PART E — VERIFY ORIGINAL RIBFRAC SOURCE ASSIGNMENTS
# ============================================================

saved_source_map = dict(zip(
    saved_split_df["patient_id"],
    saved_split_df["source_partition"]
))

assert set(saved_source_map) == set(patient_source_map)

for patient_id, source_name in saved_source_map.items():

    assert source_name == patient_source_map[patient_id], (
        f"Source mismatch for {patient_id}"
    )

print("Original source assignment consistency: PASS")


# ============================================================
# PART F — DISPLAY SPLIT DISTRIBUTION
# ============================================================

print("\n" + "-" * 65)
print("DATASET PARTITION SUMMARY")
print("-" * 65)

split_summary = (
    saved_split_df["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .rename_axis("split")
    .reset_index(name="patients")
)

split_summary["percentage"] = (
    split_summary["patients"] / 500 * 100
).round(2)

display(split_summary)

print("\nOriginal RibFrac source distribution:")

source_distribution = pd.crosstab(
    saved_split_df["split"],
    saved_split_df["source_partition"]
)

display(source_distribution)


# ============================================================
# PART G — FINAL VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("FINAL PATIENT-LEVEL LEAKAGE VERIFICATION")
print("=" * 65)

print("Training patients:", len(train_set))
print("Validation patients:", len(val_set))
print("Testing patients:", len(test_set))

print("Total unique patients:", len(all_split_patients))
print("Overlapping patients: 0")
print("Missing patients: 0")

print("\nSaved split:", split_path)

print("\nCELL #26 VALIDATION: PASS")
print("No patient-level overlap detected.")


RIBFRAC PATIENT-LEVEL DATA LEAKAGE VERIFICATION

Patient counts:
Training: 350
Validation: 75
Testing: 75

Patient counts and uniqueness: PASS

-----------------------------------------------------------------
PATIENT OVERLAP ANALYSIS
-----------------------------------------------------------------
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0

Patient overlap verification: PASS

-----------------------------------------------------------------
PATIENT COVERAGE
-----------------------------------------------------------------
Metadata patients: 500
Assigned patients: 500
Missing patients: 0
Unexpected patients: 0

Patient coverage verification: PASS

Saved split consistency: PASS
Original source assignment consistency: PASS

-----------------------------------------------------------------
DATASET PARTITION SUMMARY
-----------------------------------------------------------------


,split,patients,percentage
0,train,350,70.0
1,validation,75,15.0
2,test,75,15.0



Original RibFrac source distribution:


source_partition,train_part1,train_part2,validation
split,,,
test,50,11,14
train,212,87,51
validation,38,22,15



FINAL PATIENT-LEVEL LEAKAGE VERIFICATION
Training patients: 350
Validation patients: 75
Testing patients: 75
Total unique patients: 500
Overlapping patients: 0
Missing patients: 0

Saved split: /kaggle/working/ribfrac_metadata/patient_split_500.csv

CELL #26 VALIDATION: PASS
No patient-level overlap detected.


## 27. Save patient split

In [41]:

# ============================================================
# CELL #27 — SAVE AND PRESERVE RIBFRAC PATIENT SPLIT
# ============================================================

# ============================================================
# PART A — CONFIGURE SPLIT OUTPUT
# ============================================================

# Use the canonical split established in Cell #25.
split_path = Path(SPLIT_PATH)

split_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 60)
print("RIBFRAC PATIENT SPLIT PRESERVATION")
print("=" * 60)

print("Canonical split file:", split_path)


# ============================================================
# PART B — VERIFY PATIENT ASSIGNMENTS
# ============================================================

required_columns = [
    "patient_id",
    "split",
    "source_partition"
]

assert all(
    column in split_df.columns
    for column in required_columns
), "Patient split contains missing columns."

verified_split_df = split_df[
    required_columns
].copy()

verified_split_df["patient_id"] = (
    verified_split_df["patient_id"].astype(str)
)

verified_split_df["split"] = (
    verified_split_df["split"].astype(str)
)

# Validate unique patients
assert len(verified_split_df) == 500

assert not verified_split_df[
    "patient_id"
].duplicated().any()

assert set(
    verified_split_df["patient_id"]
) == set(patient_ids)

# Validate split sizes
expected_counts = {
    "train": 350,
    "validation": 75,
    "test": 75
}

actual_counts = (
    verified_split_df["split"].value_counts()
)

for name, expected in expected_counts.items():

    actual = int(actual_counts.get(name, 0))

    assert actual == expected, (
        f"{name}: Expected {expected}, found {actual}"
    )

assert set(actual_counts.index) == set(expected_counts)

# Verify original RibFrac source assignments
assert all(
    patient_source_map[patient_id] == source
    for patient_id, source in zip(
        verified_split_df["patient_id"],
        verified_split_df["source_partition"]
    )
)

print("\nPatient assignment verification: PASS")


# ============================================================
# PART C — SORT FOR REPRODUCIBILITY
# ============================================================

verified_split_df = (
    verified_split_df
    .sort_values("patient_id")
    .reset_index(drop=True)
)


# ============================================================
# PART D — SAVE OR VERIFY EXISTING FILE
# ============================================================

if split_path.exists():

    saved_df = pd.read_csv(split_path)

    saved_df = (
        saved_df[required_columns]
        .sort_values("patient_id")
        .reset_index(drop=True)
    )

    if not saved_df.equals(verified_split_df):

        raise RuntimeError(
            "Saved patient assignments differ from "
            "the verified assignments. "
            "The existing file was not overwritten."
        )

    print("\nExisting patient split verified.")
    print("No overwrite was necessary.")

else:

    verified_split_df.to_csv(
        split_path,
        index=False
    )

    print("\nPatient split saved successfully.")


# ============================================================
# PART E — CREATE REPRODUCIBILITY FINGERPRINT
# ============================================================

# A fingerprint helps identify changes in the exact
# patient assignments across notebooks.

split_bytes = verified_split_df.to_csv(
    index=False
).encode("utf-8")

split_sha256 = hashlib.sha256(
    split_bytes
).hexdigest()

print("\nSplit SHA256 fingerprint:")
print(split_sha256)


# ============================================================
# PART F — DISPLAY FINAL SPLIT SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL PATIENT SPLIT SUMMARY")
print("=" * 60)

split_summary = (
    verified_split_df["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print(split_summary)

print("\nSource distribution:")

display(
    pd.crosstab(
        verified_split_df["split"],
        verified_split_df["source_partition"]
    )
)

print("\nFirst 10 patient assignments:")
display(verified_split_df.head(10))


# ============================================================
# PART G — FINAL FILE VALIDATION
# ============================================================

assert split_path.is_file()

final_saved_df = pd.read_csv(split_path)

assert len(final_saved_df) == 500

assert not final_saved_df[
    "patient_id"
].duplicated().any()

print("\n" + "=" * 60)
print("CELL #27 VALIDATION: PASS")
print("=" * 60)

print("Total patients:", len(final_saved_df))
print("Split file:", split_path)
print("Assignments preserved successfully.")


RIBFRAC PATIENT SPLIT PRESERVATION
Canonical split file: /kaggle/working/ribfrac_metadata/patient_split_500.csv

Patient assignment verification: PASS

Existing patient split verified.
No overwrite was necessary.

Split SHA256 fingerprint:
3ba488e73bffcfb94ed3f4532e4cd2a2b1c14ce992a0e70467a1981e92bf4a06

FINAL PATIENT SPLIT SUMMARY
split
train         350
validation     75
test           75
Name: count, dtype: int64

Source distribution:


source_partition,train_part1,train_part2,validation
split,,,
test,50,11,14
train,212,87,51
validation,38,22,15



First 10 patient assignments:


,patient_id,split,source_partition
0,RibFrac1,test,train_part1
1,RibFrac10,train,train_part1
2,RibFrac100,test,train_part1
3,RibFrac101,train,train_part1
4,RibFrac102,train,train_part1
5,RibFrac103,validation,train_part1
6,RibFrac104,train,train_part1
7,RibFrac105,train,train_part1
8,RibFrac106,train,train_part1
9,RibFrac107,test,train_part1



CELL #27 VALIDATION: PASS
Total patients: 500
Split file: /kaggle/working/ribfrac_metadata/patient_split_500.csv
Assignments preserved successfully.


## 28. Create a final patient manifest

In [42]:

# ============================================================
# CELL #28 — CREATE FINAL RIBFRAC PATIENT MANIFEST
# ============================================================

# ============================================================
# PART A — VERIFY REQUIRED DATA
# ============================================================

print("=" * 65)
print("CREATING FINAL RIBFRAC PATIENT MANIFEST")
print("=" * 65)

assert len(patient_registry) == 500
assert len(split_df) == 500
assert len(ct_zip_index) == 500
assert len(label_index_df) == 500

assert info_df["public_id"].nunique() == 500

# ============================================================
# PART B — CREATE MASTER PATIENT REGISTRY
# ============================================================

manifest = patient_registry[
    ["public_id", "source_partition"]
].copy()

manifest = manifest.rename(
    columns={"public_id": "patient_id"}
)

assert manifest["patient_id"].nunique() == 500
assert not manifest["patient_id"].duplicated().any()

# ============================================================
# PART C — ATTACH TRAIN/VALIDATION/TEST ASSIGNMENTS
# ============================================================

split_data = split_df[
    ["patient_id", "source_partition", "split"]
].copy()

manifest = manifest.merge(
    split_data,
    on=["patient_id", "source_partition"],
    how="left",
    validate="one_to_one"
)

assert manifest["split"].notna().all(), (
    "Some patients have missing split assignments."
)

expected_split_counts = {
    "train": 350,
    "validation": 75,
    "test": 75
}

assert (
    manifest["split"].value_counts().to_dict()
    == expected_split_counts
), "Patient split distribution is incorrect."

# ============================================================
# PART D — ATTACH CT IMAGE INDEX
# ============================================================

ct_data = ct_zip_index[
    [
        "patient_id",
        "source_partition",
        "filename",
        "compressed_size",
        "uncompressed_size",
        "local_header_offset",
        "compression_method",
        "crc32"
    ]
].copy()

ct_data = ct_data.rename(
    columns={
        "filename": "ct_member",
        "compressed_size": "ct_compressed_size",
        "uncompressed_size": "ct_uncompressed_size",
        "local_header_offset": "ct_local_header_offset",
        "compression_method": "ct_compression_method",
        "crc32": "ct_crc32"
    }
)

manifest = manifest.merge(
    ct_data,
    on=["patient_id", "source_partition"],
    how="left",
    validate="one_to_one"
)

# Original CT archive URL
manifest["ct_archive_url"] = (
    manifest["source_partition"]
    .map(CT_ZIP_URLS)
)

# Indicates whether the CT is indexed.
# It does not mean the CT has been downloaded.
manifest["has_ct"] = (
    manifest["ct_member"].notna()
    & manifest["ct_archive_url"].notna()
)

# ============================================================
# PART E — ATTACH FRACTURE ANNOTATION INDEX
# ============================================================

label_data = label_index_df[
    [
        "patient_id",
        "source_partition",
        "archive_path",
        "label_member",
        "label_filename"
    ]
].copy()

label_data = label_data.rename(
    columns={
        "archive_path": "label_archive_path"
    }
)

manifest = manifest.merge(
    label_data,
    on=["patient_id", "source_partition"],
    how="left",
    validate="one_to_one"
)

manifest["has_label"] = (
    manifest["label_member"].notna()
    & manifest["label_archive_path"].notna()
)

# ============================================================
# PART F — ADD PATIENT ANNOTATION SUMMARY
# ============================================================

annotation_data = patient_annotations[
    [
        "public_id",
        "source_partition",
        "number_of_annotations",
        "number_of_fractures",
        "number_of_unknown_subtypes",
        "number_of_classified_fractures"
    ]
].copy()

annotation_data = annotation_data.rename(
    columns={"public_id": "patient_id"}
)

manifest = manifest.merge(
    annotation_data,
    on=["patient_id", "source_partition"],
    how="left",
    validate="one_to_one"
)

# ============================================================
# PART G — VALIDATE COMPLETE MANIFEST
# ============================================================

assert len(manifest) == 500

assert manifest["patient_id"].nunique() == 500

assert not manifest["patient_id"].duplicated().any()

assert set(manifest["patient_id"]) == set(patient_ids)

# Verify CT image indexing
assert manifest["has_ct"].all(), (
    "Some patients have no CT index entry."
)

# Verify fracture annotation indexing
assert manifest["has_label"].all(), (
    "Some patients have no annotation index entry."
)

# Verify the three original sources
expected_source_counts = {
    "train_part1": 300,
    "train_part2": 120,
    "validation": 80
}

assert (
    manifest["source_partition"]
    .value_counts()
    .to_dict()
    == expected_source_counts
)

# Verify patient source consistency
assert all(
    patient_source_map[pid] == source
    for pid, source in zip(
        manifest["patient_id"],
        manifest["source_partition"]
    )
)

# Verify annotation counts
assert manifest[
    "number_of_fractures"
].notna().all()

# Verify local label ZIP archives exist
label_archives = (
    manifest["label_archive_path"]
    .dropna()
    .unique()
)

for archive_path in label_archives:

    assert Path(archive_path).is_file(), (
        f"Missing label archive: {archive_path}"
    )

# Verify canonical saved patient split
saved_split = pd.read_csv(SPLIT_PATH)

saved_assignments = dict(zip(
    saved_split["patient_id"].astype(str),
    saved_split["split"].astype(str)
))

manifest_assignments = dict(zip(
    manifest["patient_id"],
    manifest["split"]
))

assert (
    saved_assignments == manifest_assignments
), "Manifest assignments differ from saved split."

print("\nPatient manifest validation: PASS")

# ============================================================
# PART H — ARRANGE MANIFEST COLUMNS
# ============================================================

manifest_columns = [
    "patient_id",
    "source_partition",
    "split",
    "ct_archive_url",
    "ct_member",
    "ct_compressed_size",
    "ct_uncompressed_size",
    "ct_local_header_offset",
    "ct_compression_method",
    "ct_crc32",
    "label_archive_path",
    "label_member",
    "label_filename",
    "has_ct",
    "has_label",
    "number_of_annotations",
    "number_of_fractures",
    "number_of_classified_fractures",
    "number_of_unknown_subtypes"
]

manifest = manifest[
    manifest_columns
].sort_values(
    "patient_id"
).reset_index(drop=True)

# ============================================================
# PART I — SAVE FINAL MANIFEST
# ============================================================

# Preserve the original filename for compatibility
# with the later Notebook 01 cells.

manifest_path = (
    METADATA_DIR / "ribfrac_patient_manifest.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)

# ============================================================
# PART J — VERIFY SAVED FILE
# ============================================================

saved_manifest = pd.read_csv(manifest_path)

assert len(saved_manifest) == 500
assert saved_manifest["patient_id"].nunique() == 500

assert saved_manifest["has_ct"].all()
assert saved_manifest["has_label"].all()

print("\n" + "=" * 65)
print("FINAL RIBFRAC PATIENT MANIFEST SUMMARY")
print("=" * 65)

print("\nTotal patients:", len(manifest))

print(
    "Patients with CT index:",
    int(manifest["has_ct"].sum())
)

print(
    "Patients with label index:",
    int(manifest["has_label"].sum())
)

print("\nOriginal RibFrac sources:")
print(
    manifest["source_partition"]
    .value_counts()
    .sort_index()
)

print("\nExperimental dataset split:")
print(
    manifest["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nCT and label availability by source:")

display(
    manifest.groupby("source_partition")[
        ["has_ct", "has_label"]
    ].sum()
)

print("\nFirst 10 manifest records:")
display(manifest.head(10))

print("\nManifest saved:")
print(manifest_path)

print("\nCELL #28 VALIDATION: PASS")


CREATING FINAL RIBFRAC PATIENT MANIFEST

Patient manifest validation: PASS

FINAL RIBFRAC PATIENT MANIFEST SUMMARY

Total patients: 500
Patients with CT index: 500
Patients with label index: 500

Original RibFrac sources:
source_partition
train_part1    300
train_part2    120
validation      80
Name: count, dtype: int64

Experimental dataset split:
split
train         350
validation     75
test           75
Name: count, dtype: int64

CT and label availability by source:


,has_ct,has_label
source_partition,,
train_part1,300,300
train_part2,120,120
validation,80,80



First 10 manifest records:


,patient_id,source_partition,split,ct_archive_url,ct_member,ct_compressed_size,ct_uncompressed_size,ct_local_header_offset,ct_compression_method,ct_crc32,label_archive_path,label_member,label_filename,has_ct,has_label,number_of_annotations,number_of_fractures,number_of_classified_fractures,number_of_unknown_subtypes
0,RibFrac1,train_part1,test,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac1-image.nii.gz,109443421,110682920,15395682268,8,3347870264,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac1-label.nii.gz,RibFrac1-label.nii.gz,True,True,3,2,2,0
1,RibFrac10,train_part1,train,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac10-image.nii.gz,118808918,120127942,14391836808,8,1552896568,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac10-label.nii.gz,RibFrac10-label.nii.gz,True,True,9,8,5,3
2,RibFrac100,train_part1,test,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac100-image.nii.gz,110652903,111890711,3435146294,8,251323667,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac100-label.nii.gz,RibFrac100-label.nii.gz,True,True,13,12,3,9
3,RibFrac101,train_part1,train,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac101-image.nii.gz,119710494,120978682,3315435713,8,3487095889,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac101-label.nii.gz,RibFrac101-label.nii.gz,True,True,5,4,3,1
4,RibFrac102,train_part1,train,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac102-image.nii.gz,141460917,141455047,3173974709,8,2274132388,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac102-label.nii.gz,RibFrac102-label.nii.gz,True,True,18,17,8,9
5,RibFrac103,train_part1,validation,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac103-image.nii.gz,118313848,119604634,3055660774,8,2164004797,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac103-label.nii.gz,RibFrac103-label.nii.gz,True,True,12,11,2,9
6,RibFrac104,train_part1,train,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac104-image.nii.gz,107626450,108886518,2948034237,8,2024215928,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac104-label.nii.gz,RibFrac104-label.nii.gz,True,True,14,13,0,13
7,RibFrac105,train_part1,train,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac105-image.nii.gz,117939886,117933519,2830094264,8,3466363441,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac105-label.nii.gz,RibFrac105-label.nii.gz,True,True,11,10,5,5
8,RibFrac106,train_part1,train,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac106-image.nii.gz,138849679,140334227,2691244498,8,3066793188,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac106-label.nii.gz,RibFrac106-label.nii.gz,True,True,13,12,1,11
9,RibFrac107,train_part1,test,https://zenodo.org/records/3893508/files/ribfr...,Part1/RibFrac107-image.nii.gz,123283616,124715514,2567960795,8,7905595,/kaggle/working/ribfrac_label_archives/ribfrac...,Part1/RibFrac107-label.nii.gz,RibFrac107-label.nii.gz,True,True,7,6,1,5



Manifest saved:
/kaggle/working/ribfrac_metadata/ribfrac_patient_manifest.csv

CELL #28 VALIDATION: PASS


## 29. Final integrity check

In [43]:

# ============================================================
# CELL #29 — FINAL RIBFRAC DATASET INTEGRITY CHECK
# ============================================================

# ============================================================
# PART A — LOAD SAVED DATASET FILES
# ============================================================

print("=" * 65)
print("RIBFRAC FINAL DATASET INTEGRITY CHECK")
print("=" * 65)

manifest_path = Path(
    METADATA_DIR / "ribfrac_patient_manifest.csv"
)

split_path = Path(SPLIT_PATH)

if not manifest_path.is_file():
    raise FileNotFoundError(manifest_path)

if not split_path.is_file():
    raise FileNotFoundError(split_path)

manifest_check = pd.read_csv(manifest_path)
split_check = pd.read_csv(split_path)

print("\nManifest:", manifest_path)
print("Patient split:", split_path)

# ============================================================
# PART B — VERIFY PATIENT COUNTS
# ============================================================

print("\n" + "-" * 65)
print("1. PATIENT COUNT VERIFICATION")
print("-" * 65)

total_patients = len(manifest_check)

unique_patients = (
    manifest_check["patient_id"].nunique()
)

assert total_patients == 500
assert unique_patients == 500

assert not manifest_check[
    "patient_id"
].duplicated().any()

assert manifest_check[
    "patient_id"
].notna().all()

metadata_ids = set(
    info_df["public_id"].astype(str)
)

manifest_ids = set(
    manifest_check["patient_id"].astype(str)
)

assert manifest_ids == metadata_ids

print("Total patient records:", total_patients)
print("Unique patients:", unique_patients)
print("Patient registry matching: PASS")

# ============================================================
# PART C — VERIFY ORIGINAL RIBFRAC SOURCE PARTITIONS
# ============================================================

print("\n" + "-" * 65)
print("2. ORIGINAL RIBFRAC SOURCE VERIFICATION")
print("-" * 65)

expected_sources = {
    "train_part1": 300,
    "train_part2": 120,
    "validation": 80
}

source_counts = (
    manifest_check["source_partition"]
    .value_counts()
    .to_dict()
)

assert source_counts == expected_sources, (
    f"Incorrect source counts: {source_counts}"
)

for row in manifest_check.itertuples(index=False):

    assert (
        patient_source_map[row.patient_id]
        == row.source_partition
    ), f"Source mismatch: {row.patient_id}"

for source_name, expected in expected_sources.items():

    print(
        f"{source_name}: "
        f"{source_counts[source_name]} / {expected}"
    )

print("Source partition verification: PASS")

# ============================================================
# PART D — VERIFY EXPERIMENTAL DATASET SPLIT
# ============================================================

print("\n" + "-" * 65)
print("3. TRAIN/VALIDATION/TEST VERIFICATION")
print("-" * 65)

expected_splits = {
    "train": 350,
    "validation": 75,
    "test": 75
}

split_counts = (
    manifest_check["split"]
    .value_counts()
    .to_dict()
)

assert split_counts == expected_splits

assert len(split_check) == 500

assert not split_check[
    "patient_id"
].duplicated().any()

saved_assignments = dict(zip(
    split_check["patient_id"].astype(str),
    split_check["split"].astype(str)
))

manifest_assignments = dict(zip(
    manifest_check["patient_id"].astype(str),
    manifest_check["split"].astype(str)
))

assert saved_assignments == manifest_assignments, (
    "Manifest and saved patient split disagree."
)

print("Training patients:", split_counts["train"])
print("Validation patients:", split_counts["validation"])
print("Testing patients:", split_counts["test"])

print("Saved split matching: PASS")

# ============================================================
# PART E — VERIFY NO PATIENT LEAKAGE
# ============================================================

print("\n" + "-" * 65)
print("4. PATIENT-LEVEL DATA LEAKAGE")
print("-" * 65)

train_ids = set(
    manifest_check.loc[
        manifest_check["split"] == "train",
        "patient_id"
    ]
)

val_ids = set(
    manifest_check.loc[
        manifest_check["split"] == "validation",
        "patient_id"
    ]
)

test_ids = set(
    manifest_check.loc[
        manifest_check["split"] == "test",
        "patient_id"
    ]
)

train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print(
    "Training / Validation overlap:",
    len(train_val_overlap)
)

print(
    "Training / Test overlap:",
    len(train_test_overlap)
)

print(
    "Validation / Test overlap:",
    len(val_test_overlap)
)

assert not train_val_overlap
assert not train_test_overlap
assert not val_test_overlap

assert len(
    train_ids | val_ids | test_ids
) == 500

print("Patient-level leakage check: PASS")

# ============================================================
# PART F — VERIFY CT IMAGE INDEX
# ============================================================

print("\n" + "-" * 65)
print("5. CT IMAGE INDEX VERIFICATION")
print("-" * 65)

assert len(ct_zip_index) == 500

assert ct_zip_index[
    "patient_id"
].nunique() == 500

ct_lookup = ct_zip_index.set_index("patient_id")

assert manifest_check["ct_member"].notna().all()
assert manifest_check["ct_archive_url"].notna().all()

for row in manifest_check.itertuples(index=False):

    ct_entry = ct_lookup.loc[row.patient_id]

    assert (
        row.ct_member == ct_entry["filename"]
    ), f"CT filename mismatch: {row.patient_id}"

    assert (
        row.source_partition
        == ct_entry["source_partition"]
    ), f"CT source mismatch: {row.patient_id}"

    assert (
        row.ct_archive_url
        == CT_ZIP_URLS[row.source_partition]
    )

    assert int(row.ct_compressed_size) > 0
    assert int(row.ct_uncompressed_size) > 0
    assert int(row.ct_local_header_offset) >= 0

print("CT files indexed:", len(ct_zip_index))
print("CT index consistency: PASS")

# ============================================================
# PART G — VERIFY FRACTURE ANNOTATION INDEX
# ============================================================

print("\n" + "-" * 65)
print("6. FRACTURE ANNOTATION INDEX VERIFICATION")
print("-" * 65)

assert len(label_index_df) == 500

assert label_index_df[
    "patient_id"
].nunique() == 500

label_lookup_df = label_index_df.set_index(
    "patient_id"
)

assert manifest_check["label_member"].notna().all()

for row in manifest_check.itertuples(index=False):

    label_entry = label_lookup_df.loc[row.patient_id]

    assert (
        row.label_member
        == label_entry["label_member"]
    ), f"Label mismatch: {row.patient_id}"

    assert (
        row.source_partition
        == label_entry["source_partition"]
    ), f"Label source mismatch: {row.patient_id}"

    assert Path(row.label_archive_path).is_file(), (
        f"Missing label ZIP: {row.patient_id}"
    )

print("Annotation files indexed:", len(label_index_df))
print("Annotation index consistency: PASS")

# ============================================================
# PART H — VERIFY FRACTURE METADATA
# ============================================================

print("\n" + "-" * 65)
print("7. FRACTURE METADATA VERIFICATION")
print("-" * 65)

annotation_columns = [
    "number_of_annotations",
    "number_of_fractures",
    "number_of_classified_fractures",
    "number_of_unknown_subtypes"
]

assert manifest_check[
    annotation_columns
].notna().all().all()

assert (
    manifest_check[annotation_columns] >= 0
).all().all()

assert (
    manifest_check["number_of_fractures"]
    ==
    manifest_check["number_of_classified_fractures"]
    + manifest_check["number_of_unknown_subtypes"]
).all()

assert (
    manifest_check["number_of_annotations"]
    >= manifest_check["number_of_fractures"]
).all()

total_fractures = int(
    manifest_check["number_of_fractures"].sum()
)

known_fractures = int(
    manifest_check[
        "number_of_classified_fractures"
    ].sum()
)

unknown_fractures = int(
    manifest_check[
        "number_of_unknown_subtypes"
    ].sum()
)

print("Total recorded fractures:", total_fractures)
print("Known subtype fractures:", known_fractures)
print("Unknown subtype fractures:", unknown_fractures)

print("Fracture metadata consistency: PASS")

# ============================================================
# PART I — DISPLAY FINAL DATASET SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FINAL RIBFRAC DATASET SUMMARY")
print("=" * 65)

summary_df = pd.DataFrame({
    "Verification": [
        "Total patients",
        "Unique patients",
        "CT files indexed",
        "Annotation files indexed",
        "Training patients",
        "Validation patients",
        "Testing patients",
        "Patient overlaps",
        "Missing patient assignments"
    ],
    "Result": [
        total_patients,
        unique_patients,
        len(ct_zip_index),
        len(label_index_df),
        len(train_ids),
        len(val_ids),
        len(test_ids),
        0,
        int(manifest_check["split"].isna().sum())
    ],
    "Status": ["PASS"] * 9
})

display(summary_df)

# ============================================================
# PART J — SAVE INTEGRITY REPORT
# ============================================================

INTEGRITY_REPORT_PATH = (
    METADATA_DIR / "ribfrac_integrity_report_500.csv"
)

summary_df.to_csv(
    INTEGRITY_REPORT_PATH,
    index=False
)

print("\nIntegrity report saved:")
print(INTEGRITY_REPORT_PATH)

# ============================================================
# PART K — FINAL VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("CELL #29 — FINAL INTEGRITY CHECK")
print("=" * 65)

print("Patient registry: PASS")
print("Source partitions: PASS")
print("Experimental split: PASS")
print("Patient leakage: PASS")
print("CT archive indexing: PASS")
print("Annotation archive indexing: PASS")
print("Fracture metadata consistency: PASS")

print("\nCELL #29 VALIDATION: PASS")
print("Dataset manifest is ready for downstream processing.")


RIBFRAC FINAL DATASET INTEGRITY CHECK

Manifest: /kaggle/working/ribfrac_metadata/ribfrac_patient_manifest.csv
Patient split: /kaggle/working/ribfrac_metadata/patient_split_500.csv

-----------------------------------------------------------------
1. PATIENT COUNT VERIFICATION
-----------------------------------------------------------------
Total patient records: 500
Unique patients: 500
Patient registry matching: PASS

-----------------------------------------------------------------
2. ORIGINAL RIBFRAC SOURCE VERIFICATION
-----------------------------------------------------------------
train_part1: 300 / 300
train_part2: 120 / 120
validation: 80 / 80
Source partition verification: PASS

-----------------------------------------------------------------
3. TRAIN/VALIDATION/TEST VERIFICATION
-----------------------------------------------------------------
Training patients: 350
Validation patients: 75
Testing patients: 75
Saved split matching: PASS

----------------------------------

,Verification,Result,Status
0,Total patients,500,PASS
1,Unique patients,500,PASS
2,CT files indexed,500,PASS
3,Annotation files indexed,500,PASS
4,Training patients,350,PASS
5,Validation patients,75,PASS
6,Testing patients,75,PASS
7,Patient overlaps,0,PASS
8,Missing patient assignments,0,PASS



Integrity report saved:
/kaggle/working/ribfrac_metadata/ribfrac_integrity_report_500.csv

CELL #29 — FINAL INTEGRITY CHECK
Patient registry: PASS
Source partitions: PASS
Experimental split: PASS
Patient leakage: PASS
CT archive indexing: PASS
Annotation archive indexing: PASS
Fracture metadata consistency: PASS

CELL #29 VALIDATION: PASS
Dataset manifest is ready for downstream processing.


## 30. Final test using RibFrac128

In [ ]:

# ============================================================
# CELL #30 — FINAL RIBFRAC PATIENT VALIDATION
# ============================================================

# ============================================================
# PART A — VERIFY REQUIRED FUNCTIONS AND DATA
# ============================================================

print("=" * 65)
print("RIBFRAC FINAL PATIENT VALIDATION")
print("=" * 65)

assert len(patient_source_map) == 500
assert len(ct_zip_index) == 500
assert info_df["public_id"].nunique() == 500

assert callable(load_patient)
assert callable(validate_patient)

print("\nDataset registry: PASS")
print("Patient loader available: PASS")
print("Patient validator available: PASS")


# ============================================================
# PART B — SELECT ONE PATIENT FROM EACH SOURCE
# ============================================================

# Use patients with positive fracture-instance metadata.
# For Part 2 and Validation, select a smaller indexed CT
# to make this final retrieval test more economical.

annotated_patients = set(
    info_df.loc[
        info_df["label_id"] > 0,
        "public_id"
    ].astype(str)
)

test_patients = {
    "train_part1": "RibFrac128"
}

assert "RibFrac128" in annotated_patients

for source_name in ["train_part2", "validation"]:

    candidates = ct_zip_index.loc[
        (ct_zip_index["source_partition"] == source_name)
        &
        (ct_zip_index["patient_id"].isin(
            annotated_patients
        ))
    ].copy()

    if candidates.empty:
        raise RuntimeError(
            f"No annotated patients in {source_name}"
        )

    candidates = candidates.sort_values(
        ["compressed_size", "patient_id"]
    )

    test_patients[source_name] = (
        str(candidates.iloc[0]["patient_id"])
    )

print("\nSelected patients:")

for source_name, patient_id in test_patients.items():
    print(f"{source_name}: {patient_id}")


# ============================================================
# PART C — DEFINE FINAL PATIENT TEST FUNCTION
# ============================================================

def run_final_patient_test(patient_id, source_name):
    """
    Load and validate one patient.

    Reuses the patient loaded in Cell #19 when possible.

    Returns a small results dictionary without retaining
    the full CT and annotation arrays.
    """

    result = {
        "patient_id": patient_id,
        "source_partition": source_name,
        "ct_shape": "",
        "label_shape": "",
        "fracture_instances": None,
        "failed_checks": "",
        "status": "NOT TESTED",
        "error": ""
    }

    loaded_patient = None

    try:

        print("\n" + "=" * 65)
        print("TESTING PATIENT:", patient_id)
        print("SOURCE:", source_name)
        print("=" * 65)

        # ----------------------------------------------------
        # 1. Load CT and annotation
        # ----------------------------------------------------

        cached_patient = globals().get("patient")

        if (
            isinstance(cached_patient, dict)
            and cached_patient.get("patient_id") == patient_id
            and "source_partition" in cached_patient
        ):

            print("Reusing patient loaded in Cell #19.")
            loaded_patient = cached_patient

        else:

            loaded_patient = load_patient(patient_id)

        # ----------------------------------------------------
        # 2. Verify source and geometry
        # ----------------------------------------------------

        assert (
            loaded_patient["patient_id"] == patient_id
        )

        assert (
            loaded_patient["source_partition"]
            == source_name
        )

        ct = loaded_patient["ct_volume"]
        label = loaded_patient["label_volume"]

        result["ct_shape"] = str(ct.shape)
        result["label_shape"] = str(label.shape)

        # ----------------------------------------------------
        # 3. Run Cell #20 patient validation
        # ----------------------------------------------------

        validation = validate_patient(
            loaded_patient
        )

        print("\nPatient validation results:")

        failed_checks = []

        for check, passed in validation.items():

            passed = bool(passed)

            print(
                f"{check:30s}: "
                f"{'PASS' if passed else 'FAIL'}"
            )

            if not passed:
                failed_checks.append(check)

        # ----------------------------------------------------
        # 4. Count fracture instances
        # ----------------------------------------------------

        fracture_ids = np.unique(
            label[label > 0]
        )

        result["fracture_instances"] = (
            len(fracture_ids)
        )

        print("\nCT shape:", ct.shape)
        print("Label shape:", label.shape)

        print(
            "Fracture instances:",
            len(fracture_ids)
        )

        # ----------------------------------------------------
        # 5. Determine final status
        # ----------------------------------------------------

        if not failed_checks:

            result["status"] = "PASS"

            print("\nPATIENT VALIDATION: PASS")

        else:

            result["status"] = "FAIL"

            result["failed_checks"] = ", ".join(
                failed_checks
            )

            print("\nPATIENT VALIDATION: FAIL")

    except Exception as error:

        result["status"] = "ERROR"
        result["error"] = (
            f"{type(error).__name__}: {error}"
        )

        print("\nPATIENT VALIDATION: ERROR")
        print(result["error"])

        traceback.print_exc(limit=2)

    finally:

        # Release locally held image arrays between tests.
        # A patient cached in Cell #19 remains available.
        del loaded_patient
        gc.collect()

    return result


# ============================================================
# PART D — RUN TESTS ACROSS ALL THREE SOURCES
# ============================================================

final_test_results = []

for source_name, patient_id in test_patients.items():

    result = run_final_patient_test(
        patient_id,
        source_name
    )

    final_test_results.append(result)


# ============================================================
# PART E — CREATE FINAL VALIDATION REPORT
# ============================================================

final_validation_df = pd.DataFrame(
    final_test_results
)

print("\n" + "=" * 65)
print("FINAL PATIENT VALIDATION REPORT")
print("=" * 65)

display(final_validation_df)


# ============================================================
# PART F — SAVE VALIDATION RESULTS
# ============================================================

FINAL_VALIDATION_PATH = (
    METADATA_DIR /
    "ribfrac_final_patient_validation.csv"
)

final_validation_df.to_csv(
    FINAL_VALIDATION_PATH,
    index=False
)

print("\nReport saved:")
print(FINAL_VALIDATION_PATH)


# ============================================================
# PART G — FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 65)
print("CELL #30 — FINAL VERIFICATION")
print("=" * 65)

passed = int(
    (final_validation_df["status"] == "PASS").sum()
)

failed = int(
    (final_validation_df["status"] != "PASS").sum()
)

print("Sources tested:", len(final_validation_df))
print("Successful patients:", passed)
print("Unsuccessful patients:", failed)

if passed == 3:

    print("\nCELL #30 VALIDATION: PASS")

    print(
        "All three RibFrac sources passed "
        "the sample end-to-end loading tests."
    )

    print(
        "\nNotebook 01 is ready for "
        "final review before Notebook 02."
    )

else:

    print("\nCELL #30 VALIDATION: REVIEW REQUIRED")

    print(
        "Investigate the unsuccessful patient tests "
        "before proceeding."
    )


RIBFRAC FINAL PATIENT VALIDATION

Dataset registry: PASS
Patient loader available: PASS
Patient validator available: PASS

Selected patients:
train_part1: RibFrac128
train_part2: RibFrac302
validation: RibFrac497

TESTING PATIENT: RibFrac128
SOURCE: train_part1
Reusing patient loaded in Cell #19.

Fracture annotation consistency:
NIfTI fracture IDs: 8
Metadata fracture IDs: 8
NIfTI IDs missing from metadata: []
Metadata IDs missing from NIfTI: []

Patient validation results:
ct_is_3d                      : PASS
label_is_3d                   : PASS
shape_match                   : PASS
spacing_match                 : PASS
affine_match                  : PASS
ct_finite                     : PASS
label_finite                  : PASS
has_annotation                : PASS
label_ids_integer             : PASS
source_match                  : PASS
metadata_available            : PASS
metadata_source_match         : PASS
nifti_ids_in_metadata         : PASS
metadata_ids_in_nifti         : PASS

C